<a href="https://colab.research.google.com/github/anamitra-tech/ML-Projects/blob/main/AegisDrone%20%E2%80%94%20AI-based%20Drone%20Threat%20Detection%20%26%20Classification%20SystemFinal4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
for f in ["dronerf_features_v28.csv", "antidrone_db_v28.json"]:
    if os.path.exists(f):
        os.remove(f)
        print(f"Deleted {f}")

In [2]:
CLASS_NAMES = {
    0: "Background RF",
    1: "AR Drone",
    2: "Bebop Drone",      # not "Bepop"
    3: "Phantom Drone"
}
BG_NAME = CLASS_NAMES[0]

FOLDER_MAP = {
    "background": 0,
    "ar drone":   1,
    "ar_drone":   1,
    "bebop":      2,
    "bepop":      2,       # typo variant
    "phantom":    3,
}

BUI_MAP = {
    "00000": 0,            # Background
    "10000": 2, "10001": 2,
    "10010": 2, "10011": 2,  # Bebop
    "10100": 1, "10101": 1,
    "10110": 1, "10111": 1,  # AR Drone
    "11000": 3, "11001": 3,
    "11010": 3,              # Phantom
}

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
ls /content/drive/MyDrive/DroneRF/DroneRF

'AR drone'/  'Background RF activites'/  'Bepop drone'/  'Phantom drone'/


In [5]:
# Run this in a notebook cell to see your actual folder structure
from pathlib import Path
root = Path("/content/drive/MyDrive/DroneRF/DroneRF")
for p in sorted(root.rglob("*"))[:30]:
    print(p)

/content/drive/MyDrive/DroneRF/DroneRF/AR drone
/content/drive/MyDrive/DroneRF/DroneRF/AR drone/RF Data_10100_H
/content/drive/MyDrive/DroneRF/DroneRF/AR drone/RF Data_10100_H.rar
/content/drive/MyDrive/DroneRF/DroneRF/AR drone/RF Data_10100_L
/content/drive/MyDrive/DroneRF/DroneRF/AR drone/RF Data_10100_L.rar
/content/drive/MyDrive/DroneRF/DroneRF/AR drone/RF Data_10101_H
/content/drive/MyDrive/DroneRF/DroneRF/AR drone/RF Data_10101_H.rar
/content/drive/MyDrive/DroneRF/DroneRF/AR drone/RF Data_10101_L
/content/drive/MyDrive/DroneRF/DroneRF/AR drone/RF Data_10101_L.rar
/content/drive/MyDrive/DroneRF/DroneRF/AR drone/RF Data_10110_H
/content/drive/MyDrive/DroneRF/DroneRF/AR drone/RF Data_10110_H.rar
/content/drive/MyDrive/DroneRF/DroneRF/AR drone/RF Data_10110_L
/content/drive/MyDrive/DroneRF/DroneRF/AR drone/RF Data_10110_L.rar
/content/drive/MyDrive/DroneRF/DroneRF/AR drone/RF Data_10111_H
/content/drive/MyDrive/DroneRF/DroneRF/AR drone/RF Data_10111_H.rar
/content/drive/MyDrive/Drone

In [6]:
!pip install rarfile

In [7]:
import os
from pathlib import Path

base = Path("/content/drive/MyDrive/DroneRF/DroneRF")

for rar_file in base.rglob("*.rar"):
    print(f"Extracting: {rar_file}")
    os.system(f'unrar x -o+ "{rar_file}" "{rar_file.parent}/"')

Extracting: /content/drive/MyDrive/DroneRF/DroneRF/Background RF activites/RF Data_00000_L1.rar
Extracting: /content/drive/MyDrive/DroneRF/DroneRF/Background RF activites/RF Data_00000_H1.rar
Extracting: /content/drive/MyDrive/DroneRF/DroneRF/Background RF activites/RF Data_00000_L2.rar
Extracting: /content/drive/MyDrive/DroneRF/DroneRF/Background RF activites/FR Data_00000_H2.rar
Extracting: /content/drive/MyDrive/DroneRF/DroneRF/Bepop drone/RF Data_10000_L.rar
Extracting: /content/drive/MyDrive/DroneRF/DroneRF/Bepop drone/RF Data_10000_H.rar
Extracting: /content/drive/MyDrive/DroneRF/DroneRF/Bepop drone/RF Data_10011_L.rar
Extracting: /content/drive/MyDrive/DroneRF/DroneRF/Bepop drone/RF Data_10011_H.rar
Extracting: /content/drive/MyDrive/DroneRF/DroneRF/Bepop drone/RF Data_10010_L.rar
Extracting: /content/drive/MyDrive/DroneRF/DroneRF/Bepop drone/RF Data_10001_L.rar
Extracting: /content/drive/MyDrive/DroneRF/DroneRF/Bepop drone/RF Data_10001_H.rar
Extracting: /content/drive/MyDrive/

In [8]:
import subprocess, sys

def force_reinstall_torch():
    print("Fixing PyTorch installation...")
    # Uninstall existing torch to avoid conflicts
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torch", "torchvision", "torchaudio"],
                   capture_output=True)
    # Reinstall CPU-only version explicitly
    subprocess.run([sys.executable, "-m", "pip", "install", "torch", "torchvision", "torchaudio",
                    "--index-url", "https://download.pytorch.org/whl/cpu", "--no-cache-dir"],
                   capture_output=True)
    print("PyTorch reinstalled.")

force_reinstall_torch()

Fixing PyTorch installation...
PyTorch reinstalled.


In [9]:
for p in sorted(root.rglob("*"))[:30]:
    print(p)

/content/drive/MyDrive/DroneRF/DroneRF/AR drone
/content/drive/MyDrive/DroneRF/DroneRF/AR drone/RF Data_10100_H
/content/drive/MyDrive/DroneRF/DroneRF/AR drone/RF Data_10100_H/10100H_0.csv
/content/drive/MyDrive/DroneRF/DroneRF/AR drone/RF Data_10100_H/10100H_1.csv
/content/drive/MyDrive/DroneRF/DroneRF/AR drone/RF Data_10100_H/10100H_10.csv
/content/drive/MyDrive/DroneRF/DroneRF/AR drone/RF Data_10100_H/10100H_11.csv
/content/drive/MyDrive/DroneRF/DroneRF/AR drone/RF Data_10100_H/10100H_12.csv
/content/drive/MyDrive/DroneRF/DroneRF/AR drone/RF Data_10100_H/10100H_13.csv
/content/drive/MyDrive/DroneRF/DroneRF/AR drone/RF Data_10100_H/10100H_14.csv
/content/drive/MyDrive/DroneRF/DroneRF/AR drone/RF Data_10100_H/10100H_15.csv
/content/drive/MyDrive/DroneRF/DroneRF/AR drone/RF Data_10100_H/10100H_16.csv
/content/drive/MyDrive/DroneRF/DroneRF/AR drone/RF Data_10100_H/10100H_17.csv
/content/drive/MyDrive/DroneRF/DroneRF/AR drone/RF Data_10100_H/10100H_18.csv
/content/drive/MyDrive/DroneRF/D

In [10]:
root = Path("/content/drive/MyDrive/DroneRF/DroneRF")
print("TOP LEVEL FOLDERS:")
for p in sorted(root.iterdir()):
    if p.is_dir():
        csv_count = len(list(p.rglob("*.csv")))
        print(f"  {p.name!r:<30} → {csv_count} CSV files")

TOP LEVEL FOLDERS:
  'AR drone'                     → 162 CSV files
  'Background RF activites'      → 82 CSV files
  'Bepop drone'                  → 168 CSV files
  'Phantom drone'                → 42 CSV files


In [11]:
%pip install -q dagshub mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/1

In [12]:
import os
import dagshub
import mlflow

# 1. Define your token
token = '99f3460b1ebc1c54e6f414e991d6f58a4cd923ae'

# 2. Add the token directly to the DagsHub auth handler
# This bypasses the need for the interactive popup
dagshub.auth.add_app_token(token)

# 3. Now initialize (this should now detect the token and NOT show the popup)
dagshub.init(repo_owner='anamitra1205', repo_name='my-first-repo', mlflow=True)

# 4. Set the experiment
mlflow.set_experiment("Drone_Detection_Training_v32")

print("✓ Connected to DagsHub!")

Accessing as anamitra1205

Initialized MLflow to track repo "anamitra1205/my-first-repo"

Repository anamitra1205/my-first-repo initialized!

✓ Connected to DagsHub!


In [13]:
import mlflow
with mlflow.start_run():
  # Your training code here...
  mlflow.log_metric('accuracy', 42)
  mlflow.log_param('Param name', 'Value')

🏃 View run gentle-moth-288 at: https://dagshub.com/anamitra1205/my-first-repo.mlflow/#/experiments/2/runs/99accaf3c4f44327bc15dbb52a55296f
🧪 View experiment at: https://dagshub.com/anamitra1205/my-first-repo.mlflow/#/experiments/2


In [14]:
import mlflow
import mlflow.sklearn
import mlflow.pytorch

# This tells MLflow to watch scikit-learn, pytorch, and others
mlflow.autolog()

2026/05/11 07:14:34 WARNING mlflow.spark: With Pyspark >= 3.2, PYSPARK_PIN_THREAD environment variable must be set to false for Spark datasource autologging to work.
2026/05/11 07:14:34 INFO mlflow.tracking.fluent: Autologging successfully enabled for pyspark.


In [15]:
# ─────────────────────────────────────────────────────────────────────────────
#  DagsHub + MLflow Integration Patch  —  v28-FIXED
#  Drop this into your Colab notebook BEFORE the SECTION 17 · MAIN block.
#  It monkey-patches build_and_evaluate to log everything to DagsHub.
# ─────────────────────────────────────────────────────────────────────────────

# STEP 0 · Install dependencies
import subprocess, sys

def _pip(*pkgs):
    for flags in [[], ["--break-system-packages"]]:
        r = subprocess.run(
            [sys.executable, "-m", "pip", "install", *pkgs, "-q", *flags],
            capture_output=True)
        if r.returncode == 0:
            return

_pip("mlflow", "dagshub")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 1 · Connect to DagsHub
# ─────────────────────────────────────────────────────────────────────────────
import mlflow
import dagshub

DAGSHUB_USERNAME = "anamitra1205"        # your DagsHub username
DAGSHUB_REPO     = "my-first-repo"       # your DagsHub repo name

dagshub.init(
    repo_owner=DAGSHUB_USERNAME,
    repo_name=DAGSHUB_REPO,
    mlflow=True,
)

mlflow.set_experiment("Drone_Detection_Training_v32")
mlflow.sklearn.autolog(disable=True)   # ← add this line

print(f"✓ DagsHub connected  →  https://dagshub.com/{DAGSHUB_USERNAME}/{DAGSHUB_REPO}")
print(f"✓ MLflow tracking URI: {mlflow.get_tracking_uri()}")


# ─────────────────────────────────────────────────────────────────────────────
# STEP 2 · Patched build_and_evaluate
#          Copy this function — it replaces the one in Section 10.
# ─────────────────────────────────────────────────────────────────────────────
def build_and_evaluate(router, X_raw_full, y, X_master, X_rf, X_gbt, X_sub, classes_present):
    """
    Drop-in replacement for the original build_and_evaluate.
    Wraps the entire training session in a single MLflow run logged to DagsHub.
    All original logic is preserved unchanged.
    """

    with mlflow.start_run(run_name="Drone_Detection_Training_v28"):

        # ── Log hyper-parameters ──────────────────────────────────────────────
        mlflow.log_params({
            # Data / windowing
            "RANDOM_SEED":       RANDOM_SEED,
            "WINDOW_SIZE":       WINDOW_SIZE,
            "STEP_SIZE":         STEP_SIZE,
            "FS":                FS,
            "TARGET_TOTAL":      TARGET_TOTAL,
            "N_FEATURES":        N_FEATURES,
            # Augmentation
            "MIXUP_ALPHA":       MIXUP_ALPHA,
            "MIXUP_N_PER_CLASS": MIXUP_N_PER_CLASS,
            "HARD_NEG_JITTER":   HARD_NEG_JITTER,
            "HARD_NEG_PCT":      HARD_NEG_PERCENTILE,
            # CNN
            "CNN_EMBED_DIM":     CNN_EMBED_DIM,
            "CNN_EPOCHS":        CNN_EPOCHS,
            "CNN_LR":            CNN_LR,
            "CNN_BATCH":         CNN_BATCH,
            "CNN_DROPOUT":       CNN_DROPOUT,
            # SVDD
            "SVDD_EMBED_DIM":    SVDD_EMBED_DIM,
            "SVDD_EPOCHS":       SVDD_EPOCHS,
            "SVDD_LR":           SVDD_LR,
            "SVDD_NU":           SVDD_NU,
            # Fusion weights
            "FUSION_W_CLF":      FUSION_W_CLF,
            "FUSION_W_CNN":      FUSION_W_CNN,
            "FUSION_W_EVM":      FUSION_W_EVM,
            "FUSION_W_NORMALITY":FUSION_W_NORMALITY,
            "FUSION_W_AGREEMENT":FUSION_W_AGREEMENT,
            # Feature selection
            "RF_TOP_K_MI":       RF_TOP_K_MI,
            "GBT_TOP_K_VAR":     GBT_TOP_K_VAR,
            # Open-set / thresholds
            "DRONE_OPEN_SET_PCT":DRONE_OPEN_SET_PERCENTILE,
            "OPEN_SET_FLOOR_PCT":OPEN_SET_FLOOR_PERCENTILE,
            "FRIENDLY_PCT":      FRIENDLY_PERCENTILE,
            "OPEN_SET_THR_CAP":  OPEN_SET_THRESHOLD_CAP,
            "HOLD_DEAD_BAND":    HOLD_DEAD_BAND,
            # Promotion
            "PROMO_MIN_OBS":     PROMO_MIN_OBS,
            "PROMO_TRUST_THR":   PROMO_TRUST_THR,
            "PROMO_MAX_THREAT":  PROMO_MAX_THREAT,
            "PROMO_CONF_THR":    PROMO_CONF_THR,
            # Build tag
            "build":             "v28-FIXED",
            "n_classes":         len(classes_present),
        })

        # ── All original training logic (unchanged) ───────────────────────────
        import numpy as np
        import time
        from sklearn.model_selection import train_test_split
        from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
        from sklearn.linear_model import LogisticRegression
        from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, average_precision_score
        from imblearn.over_sampling import SMOTE

        rng_aug = np.random.default_rng(RANDOM_SEED + 1)

        idx_tr, idx_te = train_test_split(
            np.arange(len(y)), test_size=0.20, stratify=y, random_state=RANDOM_SEED)

        X_tr_raw = X_raw_full[idx_tr]; y_tr = y[idx_tr]
        X_te_raw = X_raw_full[idx_te]; y_te = y[idx_te]

        X_tr_aug, y_tr_aug = mixup_augment(X_tr_raw, y_tr, rng_aug)

        def _scale_aug(X_aug, sc, idx):
            return np.nan_to_num(sc.transform(X_aug[:, idx]), nan=0., posinf=0., neginf=0.)

        X_m_aug  = _scale_aug(X_tr_aug, router.scaler_master, router.master_idx)
        X_rf_aug = _scale_aug(X_tr_aug, router.scaler_rf,     router.rf_idx)
        X_gb_aug = _scale_aug(X_tr_aug, router.scaler_gbt,    router.gbt_idx)
        X_sb_aug = _scale_aug(X_tr_aug, router.scaler_sub,    router.sub_idx)

        X_te_rf  = X_rf[idx_te];  X_te_gbt = X_gbt[idx_te]
        X_te_sub = X_sub[idx_te]; X_te_m   = X_master[idx_te]

        _, cnts = np.unique(y_tr_aug, return_counts=True)
        k_sm = max(1, min(5, int(cnts.min()) - 1))
        def _smote(X, y_): return SMOTE(random_state=RANDOM_SEED, k_neighbors=k_sm).fit_resample(X, y_)
        X_sm_m,  y_sm_m  = _smote(X_m_aug,  y_tr_aug)
        X_sm_rf, y_sm_rf = _smote(X_rf_aug, y_tr_aug)
        X_sm_gb, y_sm_gb = _smote(X_gb_aug, y_tr_aug)
        X_sm_sb, y_sm_sb = _smote(X_sb_aug, y_tr_aug)
        print(f"  SMOTE: master={X_sm_m.shape[0]:,}  RF={X_sm_rf.shape[0]:,}  GBT={X_sm_gb.shape[0]:,}")

        print(f"\n  [A2] Training 1D-CNN ...")
        cnn = CNNExtractor(n_classes=len(classes_present))
        cnn.fit(X_tr_aug, y_tr_aug)

        rf = RandomForestClassifier(500, class_weight="balanced", max_features="sqrt",
             min_samples_leaf=3, random_state=RANDOM_SEED, n_jobs=-1, oob_score=True)
        rf.fit(X_sm_rf, y_sm_rf)
        yp_rf  = rf.predict(X_te_rf)
        acc_rf = accuracy_score(y_te, yp_rf)
        f1_rf  = f1_score(y_te, yp_rf, average="macro", zero_division=0)
        print(f"\n  [A] RF  acc={acc_rf:.4f}  F1={f1_rf:.4f}  OOB={rf.oob_score_:.4f}")

        print(f"\n  [A1] Hard-negative mining ...")
        X_tr_hn, y_tr_hn = hard_negative_mine(X_tr_aug, y_tr_aug, rf, router.scaler_rf, router.rf_idx, rng_aug)
        if len(X_tr_hn) > len(X_tr_aug):
            X_hn_rf = _scale_aug(X_tr_hn, router.scaler_rf, router.rf_idx)
            X_sm_rf2, y_sm_rf2 = _smote(X_hn_rf, y_tr_hn)
            rf.fit(X_sm_rf2, y_sm_rf2)
            yp_rf  = rf.predict(X_te_rf)
            acc_rf = accuracy_score(y_te, yp_rf)
            f1_rf  = f1_score(y_te, yp_rf, average="macro", zero_division=0)
            print(f"  [A] RF (post-HNM) acc={acc_rf:.4f}  F1={f1_rf:.4f}")
            X_hn_gb = _scale_aug(X_tr_hn, router.scaler_gbt,   router.gbt_idx)
            X_hn_m  = _scale_aug(X_tr_hn, router.scaler_master, router.master_idx)
            X_sm_gb, y_sm_gb = _smote(X_hn_gb, y_tr_hn)
            X_sm_m,  y_sm_m  = _smote(X_hn_m,  y_tr_hn)

        gbt = GradientBoostingClassifier(n_estimators=200, learning_rate=0.08, max_depth=5,
              subsample=0.8, min_samples_leaf=5, random_state=RANDOM_SEED)
        t0 = time.time(); gbt.fit(X_sm_gb, y_sm_gb)
        yp_gbt  = gbt.predict(X_te_gbt)
        acc_gbt = accuracy_score(y_te, yp_gbt)
        f1_gbt  = f1_score(y_te, yp_gbt, average="macro", zero_division=0)
        print(f"  [B] GBT  acc={acc_gbt:.4f}  F1={f1_gbt:.4f}  ({time.time()-t0:.1f}s)")

        lr_clf = LogisticRegression(C=1.0, class_weight="balanced", max_iter=1000,
                 random_state=RANDOM_SEED, n_jobs=-1)
        lr_clf.fit(X_sm_m, y_sm_m)
        yp_lr  = lr_clf.predict(X_te_m)
        acc_lr = accuracy_score(y_te, yp_lr)
        f1_lr  = f1_score(y_te, yp_lr, average="macro", zero_division=0)
        print(f"  [C] LR   acc={acc_lr:.4f}  F1={f1_lr:.4f}")

        print(f"\n  [D] Ensemble Uncertainty:")
        ens = EnsembleUncertainty().fit(X_sm_m, y_sm_m)
        ens_p, ens_ep, _ = ens.predict_with_uncertainty(X_te_m)
        yp_ens  = ens_p.argmax(1)
        acc_ens = accuracy_score(y_te, yp_ens)
        f1_ens  = f1_score(y_te, yp_ens, average="macro", zero_division=0)
        print(f"  [D] Ensemble acc={acc_ens:.4f}  F1={f1_ens:.4f}")

        print(f"\n  [E] Phantom/AR sub-classifier:")
        sub_clf = PhantomARSubClassifier().fit(X_sm_sb, y_sm_sb)

        idx_tr2, idx_val_i = train_test_split(
            np.arange(len(idx_tr)), test_size=0.15, stratify=y[idx_tr], random_state=RANDOM_SEED)
        X_rf_val  = X_rf[idx_tr][idx_val_i]; y_rf_val = y[idx_tr][idx_val_i]
        rf_val_proba = rf.predict_proba(X_rf_val)
        ts_cal  = TemperatureScaler().fit(np.log(rf_val_proba.clip(1e-9, 1)), y_rf_val)
        cal_p   = ts_cal.calibrate(np.log(rf.predict_proba(X_te_rf).clip(1e-9, 1)))
        ece     = ts_cal.expected_calibration_error(cal_p, y_te)
        print(f"  ECE (RF, test)={ece:.4f}")

        rf_proba_te = rf.predict_proba(X_te_rf)
        roc_per_class = {}; ap_per_class = {}
        for i, cn in enumerate(classes_present):
            y_bin = (y_te == i).astype(int)
            if y_bin.sum() > 0 and y_bin.sum() < len(y_bin):
                auc = roc_auc_score(y_bin, rf_proba_te[:, i])
                ap  = average_precision_score(y_bin, rf_proba_te[:, i])
                roc_per_class[cn] = auc; ap_per_class[cn] = ap
                print(f"    {cn:<16}  ROC-AUC={auc:.4f}  AP={ap:.4f}")

        # ── Log final metrics to MLflow ───────────────────────────────────────
        mlflow.log_metrics({
            # Random Forest
            "rf_accuracy":          round(acc_rf,  4),
            "rf_f1_macro":          round(f1_rf,   4),
            "rf_oob_score":         round(rf.oob_score_, 4),
            # GBT
            "gbt_accuracy":         round(acc_gbt, 4),
            "gbt_f1_macro":         round(f1_gbt,  4),
            # Logistic Regression
            "lr_accuracy":          round(acc_lr,  4),
            "lr_f1_macro":          round(f1_lr,   4),
            # Ensemble
            "ens_accuracy":         round(acc_ens, 4),
            "ens_f1_macro":         round(f1_ens,  4),
            "ens_mean_epistemic":   round(float(ens_ep.mean()), 4),
            # Calibration
            "ece_rf":               round(ece, 4),
            "temperature_scaler_T": round(ts_cal.T, 4),
            # Per-class AUC / AP
            **{f"roc_auc_{k.replace(' ','_')}": round(v, 4) for k, v in roc_per_class.items()},
            **{f"ap_{k.replace(' ','_')}":      round(v, 4) for k, v in ap_per_class.items()},
        })

        # ── Log diagnostic images (only if they exist on disk) ───────────────
        import os
        diag_images = [
            f"{DIAG_DIR}/calibration_curves.png",
            f"{DIAG_DIR}/shap_openset_drones.png",
            f"{DIAG_DIR}/openset_confusion.png",
            f"{DIAG_DIR}/memory_hit_rate.png",
        ]
        for img_path in diag_images:
            if os.path.exists(img_path):
                mlflow.log_artifact(img_path, artifact_path="diagnostics")
                print(f"  ✓ Artifact logged → {img_path}")

        print("\n  ✓ Training run logged to DagsHub!")

        # ── Return the same dict as the original function ─────────────────────
        return {
            "rf": rf, "gbt": gbt, "lr": lr_clf, "ens": ens,
            "sub_clf": sub_clf, "ts": ts_cal, "cnn": cnn,
            "X_te_m": X_te_m, "y_te": y_te,
            "X_te_rf": X_te_rf, "y_te_rf": y_te,
            "X_te_gbt": X_te_gbt, "y_te_gbt": y_te,
            "X_te_sub": X_te_sub, "y_te_sub": y_te,
            "X_te_raw": X_te_raw,
            "X_sm_m": X_sm_m, "y_sm": y_sm_m,
            "X_sm_sub": X_sm_sb, "y_sm_sub": y_sm_sb,
            "acc_rf": acc_rf, "f1_rf": f1_rf,
            "acc_gbt": acc_gbt, "f1_gbt": f1_gbt,
            "acc_lr": acc_lr, "f1_lr": f1_lr,
            "acc_ens": acc_ens, "f1_ens": f1_ens,
            "mean_ens_ep": float(ens_ep.mean()), "ece": ece,
            "rf_proba_te": rf_proba_te, "y_te_rf": y_te,
        }

Initialized MLflow to track repo "anamitra1205/my-first-repo"

Repository anamitra1205/my-first-repo initialized!

✓ DagsHub connected  →  https://dagshub.com/anamitra1205/my-first-repo
✓ MLflow tracking URI: https://dagshub.com/anamitra1205/my-first-repo.mlflow


In [16]:

# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  v32-FIELD  —  CONSOLIDATED  (v31-FIELD + SITL PROTOTYPE ADDITIONS)        ║
# ║                                                                              ║
# ║  CHANGES vs v31-FIELD:                                                      ║
# ║                                                                              ║
# ║  [NEW-1] ActionController                                                   ║
# ║    Software-Defined "Jammer" interface — Layer 3 of the AI-Harness.        ║
# ║    MockJammer logs to defense_log.txt and prints to console.               ║
# ║    Replace MockJammer with RealHardwareController for live deployment.      ║
# ║    Triggered automatically from classify_signal when label ==               ║
# ║    POTENTIAL_THREAT or CONFIRMED_THREAT.                                   ║
# ║                                                                              ║
# ║  [NEW-2] LiveStreamSimulator                                                ║
# ║    "Virtual SDR" — reads existing .csv files and drip-feeds one burst       ║
# ║    every STREAM_INTERVAL_MS into /live_stream/. classify_signal polls       ║
# ║    that folder so the full stack runs as if a real antenna were present.   ║
# ║    run_live_stream_demo() exercises this end-to-end.                        ║
# ║                                                                              ║
# ║  [NEW-3] TemporalTracker Audit Logging                                      ║
# ║    classify_signal now prints and JSON-logs every Trust state transition:   ║
# ║      ID | Label | Seen | Trust | Trustworthy                               ║
# ║    test_tracker_stability() verifies the tracker promotes after ≥4 obs.    ║
# ║                                                                              ║
# ║  Architecture (3-layer AI-Harness):                                         ║
# ║    Layer 1 · Perception  — v31 RF classification stack (CNN/LGB/SVDD)      ║
# ║    Layer 2 · Reasoning   — SoftFusionEngine + TemporalTracker              ║
# ║    Layer 3 · Action      — ActionController → MockJammer / HW interface    ║
# ║                                                                              ║
# ║  ALL v31-FIELD pillars carried forward unchanged:                           ║
# ║  [FIX-1]  Route cache + RF fast-path @ 0.97                               ║
# ║  [FIX-2]  Open-set p10/p45 anchor, 0.10 gap, HOLD_DEAD_BAND floor         ║
# ║  [FIX-3]  preseed_fingerprint_db() — NO reset before eval                 ║
# ║  [FIX-4]  StackingMetaLearner (LR on RF+GBT+GBP probs)                   ║
# ║  [FIX-5]  LightGBM replaces sklearn RF/GBT; CNN/SVDD CUDA; TensorRT stub  ║
# ║  [FIX-6]  TRUST_MAX_VARIANCE 0.90; PRESEED_N_PER_CLASS 80                 ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# ─────────────────────────────────────────────────────────────────────────────
# HOW TO USE
# ─────────────────────────────────────────────────────────────────────────────
# Standard run (same as v31):
#   python antidrone_v32.py
#   Or in Colab: run_v32_main()
#
# Live-stream simulation (Virtual SDR):
#   run_live_stream_demo()          # streams synthetic bursts into /live_stream/
#   run_live_stream_demo(csv_dir="path/to/DroneRF")   # streams real CSVs
#
# Tracker stability unit-test:
#   test_tracker_stability()
#
# GPU acceleration (optional, auto-detects):
#   pip install lightgbm --install-option=--gpu
#   pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121
#
# TensorRT export (production only):
#   Set EXPORT_TENSORRT = True before running
# ─────────────────────────────────────────────────────────────────────────────

import subprocess, sys, os

def _pip(*pkgs):
    for flags in [[], ["--break-system-packages"]]:
        r = subprocess.run(
            [sys.executable, "-m", "pip", "install", *pkgs, "-q", *flags],
            capture_output=True)
        if r.returncode == 0:
            return

_pip("numpy", "pandas", "scipy", "scikit-learn", "imbalanced-learn",
     "matplotlib", "seaborn", "tqdm", "shap", "lightgbm")

try:
    _pip("torch", "--index-url", "https://download.pytorch.org/whl/cpu")
except Exception:
    pass

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 0 · CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────
DATA_DIR   = "/content/drive/MyDrive/DroneRF/DroneRF"
OUTPUT_CSV = "dronerf_features_v32.csv"
DB_PATH    = "antidrone_db_v32.json"
LOG_PATH   = "antidrone_audit_v32.jsonl"
DIAG_DIR   = "diagnostics_v32"

PRODUCTION_MODE = False

EXPORT_TENSORRT = False

RANDOM_SEED  = 42
WINDOW_SIZE  = 8192
STEP_SIZE    = 4096
FS           = 10e6
TARGET_TOTAL = 8000

# ── [NEW-2] Live-stream configuration ────────────────────────────────────────
LIVE_STREAM_DIR      = "live_stream"     # folder polled by the classifier
STREAM_INTERVAL_MS   = 50               # one burst every 50 ms (20 Hz)
STREAM_MAX_BURSTS    = 40               # how many bursts the demo emits

# Fusion weights (must sum to 1.0)
FUSION_W_CLF        = 0.50
FUSION_W_CNN        = 0.05
FUSION_W_EVM        = 0.20
FUSION_W_NORMALITY  = 0.15
FUSION_W_AGREEMENT  = 0.10
assert abs(FUSION_W_CLF + FUSION_W_CNN + FUSION_W_EVM +
           FUSION_W_NORMALITY + FUSION_W_AGREEMENT - 1.0) < 1e-9

HOLD_DEAD_BAND      = 0.050
MIN_HOLD_RATE       = 0.045
HOLD_CLF_PROB_LOW   = 0.88
HOLD_CLF_PROB_HIGH  = 0.90

OPEN_SET_THRESHOLD_CAP      = 0.65
DRONE_OPEN_SET_PERCENTILE   = 10.0
OPEN_SET_FLOOR_PERCENTILE   = 2
FRIENDLY_PERCENTILE         = 45
FRIENDLY_MIN_GAP            = 0.10

CONFIDENCE_BYPASS_THRESHOLD     = 0.999999
CONFIDENCE_BYPASS_THREAT_RATIO  = 0.20
BYPASS_MIN_SEEN_COUNT           = 5

HYSTERESIS_WINDOW   = 5
HYSTERESIS_MAJORITY = 6

SVDD_EMBED_DIM  = 8
SVDD_EPOCHS     = 40
SVDD_LR         = 1e-3
SVDD_BATCH      = 128
SVDD_WARMUP     = 10
SVDD_NU         = 0.01

PROMO_MIN_OBS        = 1
PROMO_TRUST_THR      = 0.20
PROMO_MAX_THREAT     = 0.85
PROMO_CONF_THR       = 0.30

GATE_RECALL_MIN       = 0.85
GATE_HOLD_MAX         = 0.20
GATE_OPEN_SET_MIN     = 0.04
GATE_FPR_MAX          = 0.10
GATE_FLICKER_MAX      = 0.65
GATE_TIME_TO_TRUST_S  = 10.0
GATE_HIT_RATE_MIN     = 0.01
GATE_BYPASS_MAX       = 0.10

MIXUP_ALPHA          = 0.30
MIXUP_N_PER_CLASS    = 800
HARD_NEG_JITTER      = 0.08
HARD_NEG_PERCENTILE  = 20

CNN_EMBED_DIM  = 16
CNN_EPOCHS     = 30
CNN_LR         = 3e-3
CNN_BATCH      = 128
CNN_DROPOUT    = 0.30

ANOMALY_W_MAHAL      = 0.55
ANOMALY_W_ISO        = 0.45
ANOMALY_SCORE_CAP    = 1.0

COST_BIAS_ACTIVE          = True
COST_BIAS_BG_PENALTY      = 0.01
COST_BIAS_UNCERTAINTY_THR = 0.55

TEMPORAL_WINDOW          = 5
TEMPORAL_SMOOTHING_MIN   = 3
TEMP_MIN = 0.70
TEMP_MAX = 1.20

TRUST_MIN_OBSERVATIONS = 4
TRUST_MAX_VARIANCE     = 0.90
HIGH_THREAT_THRESHOLD  = 0.90
CONFIRMED_THREAT_OBS   = 5
AUTO_CLASSIFY_CONF     = 0.75
HOLD_STABILITY_WINDOW  = 3

LGB_RF_N_ESTIMATORS    = 500
LGB_RF_NUM_LEAVES      = 63
LGB_RF_MIN_DATA_LEAF   = 3
LGB_RF_SUBSAMPLE       = 0.8
LGB_RF_COLSAMPLE       = 0.5

LGB_GBT_N_ESTIMATORS   = 200
LGB_GBT_NUM_LEAVES     = 31
LGB_GBT_LR             = 0.08
LGB_GBT_MIN_DATA_LEAF  = 5
LGB_GBT_SUBSAMPLE      = 0.8

_LGB_DEVICE = "cpu"

N_ENSEMBLE_TREES   = 3
ENSEMBLE_SUBSAMPLE = 0.70

SUBCLF_FEATURES = [
    "high_low_band_ratio", "spectral_centroid", "bandwidth_hz",
    "energy_band3", "energy_band4", "energy_band1", "energy_band2",
    "ifreq_std", "spectral_entropy", "tx_rate_hz", "encryption_flag",
    "freq_hop_count", "speed_mean", "altitude_mean",
]

RF_TOP_K_MI   = 45
GBT_TOP_K_VAR = 40

OCSVM_NU      = 0.05
OCSVM_GAMMA   = "scale"
HASH_N_BINS          = 20
HASH_CLIP            = 50.0
HASH_TOP_FEATURES    = 12
SIMILARITY_THRESHOLD = 0.88

GBP_TEMPERATURE         = 0.85
LAPLACE_PRIOR_PRECISION = 1.0
LAPLACE_N_SAMPLES       = 256
ISO_N_ESTIMATORS        = 300
ISO_CONTAMINATION       = 0.02
MONITOR_WINDOW          = 100

GHOST_HUNT_BURSTS    = 60
ADVERSARIAL_SAMPLES  = 200
RECOVERY_BURST_COUNT = 20

ROUTE_CACHE_MAXSIZE    = 1024
RF_FAST_PATH_THRESHOLD = 0.97
PRESEED_N_PER_CLASS    = 80

SYSTEM_LIMITATIONS = {
    "Overlapping RF signatures":
        "AR Drone 2.4GHz and Phantom 5.8GHz share band under congestion.",
    "Adversarial signals":
        "Engineered signals mimicking training statistics would evade detection.",
    "Noisy RF environments":
        "Low SNR conditions degrade spectral feature quality.",
    "Unseen drone types":
        "Novel models not in training data are flagged OPEN_SET_UNKNOWN.",
    "Simultaneous multi-drone":
        "Mixed signatures may fall outside all training distributions.",
    "Wind / battery / distance variance":
        "[FIX-6] Handled via TRUST_MAX_VARIANCE=0.90; extreme cases still HOLD.",
}

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 1 · IMPORTS
# ─────────────────────────────────────────────────────────────────────────────
import gc, hashlib, json, logging, os, re, shutil, threading, time, warnings
from collections import Counter, defaultdict, deque
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Any

import numpy as np
import pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from scipy.stats    import kurtosis, skew
from scipy.signal   import hilbert, welch, stft
from scipy.linalg   import cho_factor, cho_solve
from scipy.optimize import minimize_scalar

from sklearn.decomposition     import PCA
from sklearn.ensemble          import IsolationForest
from sklearn.feature_selection import mutual_info_classif
from sklearn.linear_model      import LogisticRegression
from sklearn.metrics           import (accuracy_score, f1_score,
                                        confusion_matrix,
                                        roc_auc_score,
                                        average_precision_score)
from sklearn.model_selection   import train_test_split
from sklearn.preprocessing     import RobustScaler
from sklearn.svm               import OneClassSVM
from imblearn.over_sampling    import SMOTE

try:
    import lightgbm as lgb
    LGB_OK = True
    print("✓ LightGBM available — fast inference path enabled")
except ImportError:
    LGB_OK = False
    print("⚠  LightGBM not available — falling back to scikit-learn RF/GBT")
    from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

if LGB_OK:
    try:
        _probe_params = {"objective": "binary", "device_type": "gpu",
                         "num_leaves": 4, "n_estimators": 1, "verbose": -1}
        _probe_data = lgb.Dataset(np.random.randn(10, 4), label=[0,1]*5)
        lgb.train(_probe_params, _probe_data, num_boost_round=1)
        _LGB_DEVICE = "gpu"
        print("✓ LightGBM GPU device available")
    except Exception:
        _LGB_DEVICE = "cpu"
        print("  LightGBM running on CPU (no GPU or CUDA not available)")

warnings.filterwarnings("ignore")
np.random.seed(RANDOM_SEED)

try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torch.utils.data import DataLoader, TensorDataset
    TORCH_OK = True
    CUDA_OK  = torch.cuda.is_available()
    DEVICE   = torch.device("cuda" if CUDA_OK else "cpu")
    if CUDA_OK:
        print(f"✓ CUDA available — CNN/SVDD running on {torch.cuda.get_device_name(0)}")
    else:
        print("✓ PyTorch CPU — 1D-CNN + Deep SVDD enabled (no CUDA)")
except ImportError:
    TORCH_OK = False
    CUDA_OK  = False
    DEVICE   = None
    print("⚠  PyTorch not available — CNN + SVDD fallback to legacy detectors")

try:
    import shap
    SHAP_OK = True
except ImportError:
    SHAP_OK = False

_audit = logging.getLogger("antidrone.v32")
_audit.setLevel(logging.DEBUG)
_fh = logging.FileHandler(LOG_PATH, mode="w")
_fh.setFormatter(logging.Formatter("%(message)s"))
_audit.addHandler(_fh)

def audit(event: str, **kw):
    _audit.debug(json.dumps({"ts": round(time.time(), 4), "event": event, **kw}))

os.makedirs(DIAG_DIR, exist_ok=True)
print(f"✓ v32-FIELD  |  Python {sys.version.split()[0]}")
print(f"  PRODUCTION_MODE = {PRODUCTION_MODE}")
print(f"  LGB device = {_LGB_DEVICE}  |  CUDA = {CUDA_OK}")

CLASS_NAMES = {0: "Background RF", 1: "AR Drone", 2: "Phantom Drone"}
BG_NAME     = CLASS_NAMES[0]
FOLDER_MAP  = {"background": 0, "ar drone": 1, "ar_drone": 1, "ardrone": 1, "phantom": 2}
BUI_MAP     = {"00000": 0, "10000": 1, "10001": 1, "10010": 1,
               "10011": 1, "10100": 1, "10101": 1, "10110": 1,
               "11000": 2, "11001": 2, "11010": 2}

DECISION_ICONS = {
    "FRIENDLY_DRONE": "🟢", "BACKGROUND": "⚪",
    "POTENTIAL_THREAT": "🔴", "CONFIRMED_THREAT": "🚨",
    "SAFE_NEW_DRONE": "🔵", "TRUSTED_NEW_DRONE": "🔷",
    "UNKNOWN_MONITOR": "🟡", "OPEN_SET_UNKNOWN": "❓", "HOLD": "⏸️",
    "MEMORY_MATCH": "💾",
}

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 2 · FEATURE SCHEMA  (83 total)
# ─────────────────────────────────────────────────────────────────────────────
RF_FEATURE_NAMES = [
    "amp_mean","amp_std","amp_var","amp_min","amp_max","amp_range",
    "amp_kurtosis","amp_skew",
    "signal_power_db","IQ_corr","I_power","Q_power","iq_power_ratio","iq_corr_sq",
    "peak_freq_hz","bandwidth_hz","spectral_entropy","spectral_centroid",
    "spectral_spread","spectral_rolloff_85","psd_mean_db","psd_max_db",
    "ifreq_mean","ifreq_std","ifreq_range","ifreq_kurtosis",
    "energy_band1","energy_band2","energy_band3","energy_band4",
    "stft_flux_var","stft_sub1_var","stft_sub2_var","stft_sub3_var","stft_sub4_var",
    "spec_kurtosis","spec_skewness","l_kurtosis","spec_flatness","stft_entropy",
    "am_depth","crest_factor","phase_jitter","spec_asymmetry",
    "acf_short","acf_medium","acf_long","acf_ratio",
    "kurt_entropy_product","snr_like_db","spectral_variance","temporal_kurtosis",
    "high_low_band_ratio",
]
FLIGHT_FEATURE_NAMES = [
    "speed_mean","speed_std","speed_max","accel_mean","accel_std","accel_max",
    "altitude_mean","altitude_std","heading_change_rate","heading_std",
    "path_curvature","loiter_fraction","approach_vector_sin","approach_vector_cos",
    "proximity_score","hover_time_fraction","trajectory_entropy","maneuver_intensity",
]
COMM_FEATURE_NAMES = [
    "tx_rate_hz","tx_burst_ratio","protocol_entropy",
    "command_interval_mean","command_interval_std","telemetry_rate_hz",
    "encryption_flag","freq_hop_count","channel_dwell_mean",
    "control_link_snr","video_link_active","swarm_signal_flag",
]
N_RF     = len(RF_FEATURE_NAMES);     assert N_RF == 53
N_FLIGHT = len(FLIGHT_FEATURE_NAMES); assert N_FLIGHT == 18
N_COMM   = len(COMM_FEATURE_NAMES);   assert N_COMM == 12
ALL_FEATURE_NAMES = RF_FEATURE_NAMES + FLIGHT_FEATURE_NAMES + COMM_FEATURE_NAMES
N_FEATURES        = len(ALL_FEATURE_NAMES)   # 83
FEAT_IDX          = {n: i for i, n in enumerate(ALL_FEATURE_NAMES)}
print(f"✓ Features: {N_RF} RF + {N_FLIGHT} flight + {N_COMM} comm = {N_FEATURES} total")

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 3 · PHYSICS-BASED SYNTHETIC DATA
# ─────────────────────────────────────────────────────────────────────────────
DRONERF_STATS = {
    0: {
        "signal_power_db":(-28.,8.),"spectral_entropy":(3.8,1.4),
        "bandwidth_hz":(0.7e6,0.5e6),"ifreq_std":(0.22,0.18),
        "amp_kurtosis":(0.6,0.9),"spectral_centroid":(2.1e6,1.0e6),
        "IQ_corr":(0.02,0.08),"crest_factor":(1.8,0.5),
        "snr_like_db":(-10.,6.),"psd_max_db":(-26.,8.),
        "energy_band1":(0.40,0.12),"energy_band2":(0.28,0.10),
        "energy_band3":(0.18,0.08),"energy_band4":(0.14,0.07),
    },
    1: {
        "signal_power_db":(-18.,6.),"spectral_entropy":(5.6,1.1),
        "bandwidth_hz":(2.2e6,0.9e6),"ifreq_std":(0.92,0.38),
        "amp_kurtosis":(2.4,1.2),"spectral_centroid":(4.5e6,0.8e6),
        "IQ_corr":(0.08,0.10),"crest_factor":(2.8,0.7),
        "snr_like_db":(8.,5.),"psd_max_db":(-16.,6.),
        "energy_band1":(0.15,0.06),"energy_band2":(0.30,0.08),
        "energy_band3":(0.35,0.09),"energy_band4":(0.20,0.07),
    },
    2: {
        "signal_power_db":(-20.,6.),"spectral_entropy":(5.2,1.1),
        "bandwidth_hz":(2.0e6,0.8e6),"ifreq_std":(0.85,0.35),
        "amp_kurtosis":(2.1,1.1),"spectral_centroid":(4.2e6,0.9e6),
        "IQ_corr":(0.07,0.09),"crest_factor":(2.6,0.6),
        "snr_like_db":(6.,4.),"psd_max_db":(-18.,6.),
        "energy_band1":(0.18,0.07),"energy_band2":(0.32,0.09),
        "energy_band3":(0.32,0.08),"energy_band4":(0.18,0.06),
    },
    3: {
        "signal_power_db":(-12.,5.5),"spectral_entropy":(6.3,0.9),
        "bandwidth_hz":(3.9e6,1.1e6),"ifreq_std":(1.58,0.48),
        "amp_kurtosis":(3.7,1.4),"spectral_centroid":(5.8e6,0.6e6),
        "IQ_corr":(0.14,0.11),"crest_factor":(3.5,0.8),
        "snr_like_db":(15.,4.),"psd_max_db":(-10.,5.),
        "energy_band1":(0.05,0.03),"energy_band2":(0.12,0.05),
        "energy_band3":(0.35,0.08),"energy_band4":(0.48,0.10),
    },
}


def _generate_rf_burst(cls: int, rng: np.random.Generator,
                        noise_scale: float = 1.0) -> np.ndarray:
    prof = DRONERF_STATS[cls]
    fv   = np.zeros(N_FEATURES, dtype=np.float32)

    def G(key, dm=0., ds=1.):
        mu, sd = prof.get(key, (dm, ds))
        return float(rng.normal(mu, sd * noise_scale))

    pwr_db=G("signal_power_db"); bw=abs(G("bandwidth_hz"))
    entr=abs(G("spectral_entropy")); ifreq=abs(G("ifreq_std"))
    kurt=G("amp_kurtosis"); cen=abs(G("spectral_centroid"))
    iq_r=G("IQ_corr"); cf=abs(G("crest_factor"))
    snr_db=G("snr_like_db"); psd_mx=G("psd_max_db")

    rms     = float(10**(pwr_db/20.))
    amp_std = rms*abs(float(rng.normal(0.35+0.05*abs(kurt),0.05)))
    amp_mean= rms*abs(float(rng.normal(1.0,0.05)))
    amp_min = max(0., amp_mean-3.*amp_std)
    amp_max = amp_mean+abs(float(rng.normal(3.5+0.3*cf,0.3)))*amp_std

    fv[FEAT_IDX["amp_mean"]]=amp_mean; fv[FEAT_IDX["amp_std"]]=amp_std
    fv[FEAT_IDX["amp_var"]]=amp_std**2; fv[FEAT_IDX["amp_min"]]=amp_min
    fv[FEAT_IDX["amp_max"]]=amp_max; fv[FEAT_IDX["amp_range"]]=amp_max-amp_min
    fv[FEAT_IDX["amp_kurtosis"]]=kurt
    fv[FEAT_IDX["amp_skew"]]=float(rng.normal(0.4*np.sign(kurt),0.2))

    i_pow=rms**2*abs(float(rng.normal(1.0,0.05)))
    q_pow=i_pow*abs(float(rng.normal(0.95+0.1*abs(iq_r),0.05)))
    fv[FEAT_IDX["signal_power_db"]]=pwr_db
    fv[FEAT_IDX["IQ_corr"]]=float(np.clip(iq_r,-0.99,0.99))
    fv[FEAT_IDX["I_power"]]=i_pow; fv[FEAT_IDX["Q_power"]]=q_pow
    fv[FEAT_IDX["iq_power_ratio"]]=i_pow/(q_pow+1e-9)
    fv[FEAT_IDX["iq_corr_sq"]]=iq_r**2

    spread=bw*abs(float(rng.normal(0.38,0.06)))
    rollof=cen+spread*abs(float(rng.normal(1.2,0.1)))
    fv[FEAT_IDX["peak_freq_hz"]]=cen+float(rng.normal(0,bw*0.05))
    fv[FEAT_IDX["bandwidth_hz"]]=bw; fv[FEAT_IDX["spectral_entropy"]]=entr
    fv[FEAT_IDX["spectral_centroid"]]=cen; fv[FEAT_IDX["spectral_spread"]]=spread
    fv[FEAT_IDX["spectral_rolloff_85"]]=rollof
    fv[FEAT_IDX["psd_mean_db"]]=pwr_db-abs(float(rng.normal(4.,1.)))
    fv[FEAT_IDX["psd_max_db"]]=psd_mx

    fv[FEAT_IDX["ifreq_mean"]]=float(rng.normal(0,ifreq*0.1))
    fv[FEAT_IDX["ifreq_std"]]=ifreq
    fv[FEAT_IDX["ifreq_range"]]=ifreq*abs(float(rng.normal(4.0,0.5)))
    fv[FEAT_IDX["ifreq_kurtosis"]]=float(rng.normal(0.5+0.3*abs(kurt),0.3))

    e1=abs(G("energy_band1")); e2=abs(G("energy_band2"))
    e3=abs(G("energy_band3")); e4=abs(G("energy_band4"))
    etot=e1+e2+e3+e4+1e-9
    b1=e1/etot; b2=e2/etot; b3=e3/etot; b4=e4/etot
    fv[FEAT_IDX["energy_band1"]]=b1; fv[FEAT_IDX["energy_band2"]]=b2
    fv[FEAT_IDX["energy_band3"]]=b3; fv[FEAT_IDX["energy_band4"]]=b4
    fv[FEAT_IDX["high_low_band_ratio"]]=(b3+b4)/(b1+b2+1e-9)

    stft_flux=bw*abs(float(rng.normal(0.01+0.005*abs(kurt),0.002)))
    fv[FEAT_IDX["stft_flux_var"]]=stft_flux
    for b in range(4):
        fv[FEAT_IDX[f"stft_sub{b+1}_var"]]=abs(
            float(rng.normal(stft_flux*(0.8+0.1*b),stft_flux*0.3)))

    fv[FEAT_IDX["spec_kurtosis"]]=float(rng.normal(kurt*0.9,0.3))
    fv[FEAT_IDX["spec_skewness"]]=float(rng.normal(0.3*np.sign(kurt),0.2))
    fv[FEAT_IDX["l_kurtosis"]]=float(rng.normal(0.2+0.05*abs(kurt),0.1))
    fv[FEAT_IDX["spec_flatness"]]=float(np.clip(rng.normal(0.5-0.04*entr,0.1),0,1))
    fv[FEAT_IDX["stft_entropy"]]=entr*abs(float(rng.normal(0.95,0.05)))
    am=np.clip(0.05+0.06*abs(kurt),0.01,0.99)
    fv[FEAT_IDX["am_depth"]]=float(am+rng.normal(0,0.02))
    fv[FEAT_IDX["crest_factor"]]=cf
    fv[FEAT_IDX["phase_jitter"]]=ifreq*abs(float(rng.normal(0.15,0.05)))
    fv[FEAT_IDX["spec_asymmetry"]]=float(rng.normal((cen-3e6)/3e6,0.1))

    acf_s=float(np.clip(rng.normal(0.1+0.05*abs(iq_r),0.05),-1,1))
    acf_m=float(np.clip(rng.normal(acf_s*0.4,0.04),-1,1))
    acf_l=float(np.clip(rng.normal(acf_m*0.3,0.03),-1,1))
    fv[FEAT_IDX["acf_short"]]=acf_s; fv[FEAT_IDX["acf_medium"]]=acf_m
    fv[FEAT_IDX["acf_long"]]=acf_l
    fv[FEAT_IDX["acf_ratio"]]=acf_s/(acf_l+1e-9)
    fv[FEAT_IDX["kurt_entropy_product"]]=float(kurt*entr)
    fv[FEAT_IDX["snr_like_db"]]=snr_db
    fv[FEAT_IDX["spectral_variance"]]=float(spread**2)
    fv[FEAT_IDX["temporal_kurtosis"]]=float(kurt+rng.normal(0,0.2))

    if cls==1:
        for k,(mu,sd) in [("speed_mean",(5.,2.)),("speed_std",(1.5,.5)),
            ("speed_max",(12.,3.)),("accel_mean",(.8,.3)),("accel_std",(.4,.15)),
            ("accel_max",(3.,.8)),("altitude_mean",(30.,15.)),("altitude_std",(5.,2.)),
            ("heading_change_rate",(.3,.1)),("trajectory_entropy",(2.5,.5)),
            ("maneuver_intensity",(.4,.15))]:
            fv[FEAT_IDX[k]]=abs(float(rng.normal(mu,sd)))
        fv[FEAT_IDX["hover_time_fraction"]]=float(np.clip(rng.normal(.25,.1),0,1))
    elif cls==2:
        for k,(mu,sd) in [("speed_mean",(12.,3.)),("speed_std",(2.5,.8)),
            ("speed_max",(22.,4.)),("accel_mean",(1.5,.4)),("accel_std",(.7,.2)),
            ("accel_max",(5.,1.)),("altitude_mean",(80.,25.)),("altitude_std",(10.,4.)),
            ("heading_change_rate",(.15,.06)),("trajectory_entropy",(3.2,.5)),
            ("maneuver_intensity",(.65,.15))]:
            fv[FEAT_IDX[k]]=abs(float(rng.normal(mu,sd)))
        fv[FEAT_IDX["hover_time_fraction"]]=float(np.clip(rng.normal(.10,.05),0,1))

    if cls==1:
        for k,v in [("tx_rate_hz",abs(float(rng.normal(25.,5.)))),
            ("tx_burst_ratio",float(np.clip(rng.normal(.35,.1),0,1))),
            ("protocol_entropy",abs(float(rng.normal(1.8,.3)))),
            ("command_interval_mean",abs(float(rng.normal(.04,.01)))),
            ("command_interval_std",abs(float(rng.normal(.008,.002)))),
            ("telemetry_rate_hz",abs(float(rng.normal(10.,2.)))),
            ("encryption_flag",0.),("freq_hop_count",abs(float(rng.normal(3.,1.)))),
            ("channel_dwell_mean",abs(float(rng.normal(.02,.005)))),
            ("control_link_snr",abs(float(rng.normal(18.,4.)))),
            ("video_link_active",float(rng.choice([0.,1.],p=[.3,.7]))),
            ("swarm_signal_flag",0.)]:
            fv[FEAT_IDX[k]]=v
    elif cls==2:
        for k,v in [("tx_rate_hz",abs(float(rng.normal(50.,8.)))),
            ("tx_burst_ratio",float(np.clip(rng.normal(.55,.12),0,1))),
            ("protocol_entropy",abs(float(rng.normal(2.5,.3)))),
            ("command_interval_mean",abs(float(rng.normal(.02,.005)))),
            ("command_interval_std",abs(float(rng.normal(.004,.001)))),
            ("telemetry_rate_hz",abs(float(rng.normal(20.,3.)))),
            ("encryption_flag",1.),("freq_hop_count",abs(float(rng.normal(8.,2.)))),
            ("channel_dwell_mean",abs(float(rng.normal(.008,.002)))),
            ("control_link_snr",abs(float(rng.normal(25.,4.)))),
            ("video_link_active",1.),
            ("swarm_signal_flag",float(rng.choice([0.,1.],p=[.85,.15])))]:
            fv[FEAT_IDX[k]]=v

    if rng.random()<0.08:
        fv[rng.integers(0,N_FEATURES,size=rng.integers(1,4))]=0.
    if rng.random()<0.05:
        fv[FEAT_IDX["amp_kurtosis"]]+=float(rng.exponential(2.))
    return fv


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 3b · AUGMENTATION
# ─────────────────────────────────────────────────────────────────────────────
def mixup_augment(X, y, rng, alpha=MIXUP_ALPHA, n_per_drone_class=MIXUP_N_PER_CLASS):
    bg_idx  = np.where(y == 0)[0]
    aug_X, aug_y = [], []
    for drone_cls in [1, 2]:
        d_idx = np.where(y == drone_cls)[0]
        if len(d_idx) == 0 or len(bg_idx) == 0:
            continue
        for _ in range(n_per_drone_class):
            di = rng.choice(d_idx); bi = rng.choice(bg_idx)
            mixed = (1. - alpha) * X[di] + alpha * X[bi]
            aug_X.append(mixed.astype(np.float32)); aug_y.append(drone_cls)
    if not aug_X: return X, y
    aug_X = np.stack(aug_X); aug_y = np.array(aug_y, dtype=np.int64)
    print(f"  [A1] Mixup: +{len(aug_X)} samples")
    return np.concatenate([X, aug_X]), np.concatenate([y, aug_y])


def hard_negative_mine(X, y, lgb_clf, scaler_rf, rf_idx, rng,
                        percentile=HARD_NEG_PERCENTILE, jitter_std=HARD_NEG_JITTER):
    drone_mask = (y != 0)
    if drone_mask.sum() < 20: return X, y
    X_drone = X[drone_mask]; y_drone = y[drone_mask]
    X_sc = np.nan_to_num(scaler_rf.transform(X_drone[:, rf_idx]), nan=0., posinf=0., neginf=0.)
    if LGB_OK and isinstance(lgb_clf, lgb.Booster):
        probs = lgb_clf.predict(X_sc)
    else:
        probs = lgb_clf.predict_proba(X_sc)
    max_p = probs.max(1); thr = np.percentile(max_p, percentile)
    hard  = max_p <= thr
    if hard.sum() == 0: return X, y
    X_hard = X_drone[hard]; y_hard = y_drone[hard]
    jittered = X_hard + rng.normal(0, jitter_std, X_hard.shape).astype(np.float32)
    print(f"  [A1] Hard-negative mining: {hard.sum()} samples jittered and added")
    return np.concatenate([X, jittered]), np.concatenate([y, y_hard])


def generate_realistic_dataset(n_per_class=2000, boundary_ratio=0.25, rng_seed=RANDOM_SEED):
    rng = np.random.default_rng(rng_seed)
    rows, labels = [], []
    for cls in range(3):
        n_normal = int(n_per_class * 0.75)
        n_noisy  = int(n_per_class * 0.15)
        n_vnoisy = n_per_class - n_normal - n_noisy
        for _ in range(n_normal):  rows.append(_generate_rf_burst(cls, rng, 1.0)); labels.append(cls)
        for _ in range(n_noisy):   rows.append(_generate_rf_burst(cls, rng, 1.6)); labels.append(cls)
        for _ in range(n_vnoisy):  rows.append(_generate_rf_burst(cls, rng, 2.5)); labels.append(cls)
    n_bnd = int(n_per_class * boundary_ratio)
    for _ in range(n_bnd):
        fv = _generate_rf_burst(1, rng, 1.2)
        fv[FEAT_IDX["spectral_centroid"]]=float(rng.normal(5.2e6,0.4e6))
        fv[FEAT_IDX["bandwidth_hz"]]=abs(float(rng.normal(3.2e6,0.8e6)))
        b3,b4=fv[FEAT_IDX["energy_band3"]],fv[FEAT_IDX["energy_band4"]]
        b1,b2=fv[FEAT_IDX["energy_band1"]],fv[FEAT_IDX["energy_band2"]]
        fv[FEAT_IDX["high_low_band_ratio"]]=(b3+b4)/(b1+b2+1e-9)
        rows.append(fv); labels.append(1)
    for _ in range(n_bnd):
        fv = _generate_rf_burst(2, rng, 1.2)
        fv[FEAT_IDX["signal_power_db"]]=float(rng.normal(-25.,3.))
        fv[FEAT_IDX["snr_like_db"]]=float(rng.normal(-8.,2.))
        rows.append(fv); labels.append(2)
    X  = np.array(rows, dtype=np.float32)
    df = pd.DataFrame(X, columns=ALL_FEATURE_NAMES)
    df.insert(0,"label_int",labels)
    df.insert(1,"label_name",[CLASS_NAMES.get(c,str(c)) for c in labels])
    df.insert(2,"source_file",["synthetic_v32"]*len(labels))
    df = df.sample(frac=1, random_state=rng_seed).reset_index(drop=True)
    cnts = Counter(labels)
    print(f"  ✓ {len(df):,} rows: "+"  ".join(f"{CLASS_NAMES.get(k,k)}={v}" for k,v in sorted(cnts.items())))
    return df


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 4 · FEATURE EXTRACTION
# ─────────────────────────────────────────────────────────────────────────────
def _pearson(x, y):
    xm=x-x.mean(); ym=y-y.mean()
    return float(np.dot(xm,ym)/((np.dot(xm,xm)*np.dot(ym,ym))**0.5+1e-12))

def extract_rf_features(real_seg, fs=FS):
    real=real_seg.astype(np.float64); N=len(real)
    analytic=hilbert(real); I,Q=analytic.real,analytic.imag
    envelope=np.abs(analytic); out=np.empty(N_RF, dtype=np.float32)
    amp_mean=float(envelope.mean()); amp_std=float(envelope.std())
    amp_min=float(envelope.min()); amp_max=float(envelope.max())
    amp_kurt=float(kurtosis(envelope)) if amp_std>1e-8 else 0.
    out[0:8]=[amp_mean,amp_std,amp_std**2,amp_min,amp_max,amp_max-amp_min,
              amp_kurt, float(skew(envelope)) if amp_std>1e-8 else 0.]
    I_pow=float(np.dot(I,I)/N); Q_pow=float(np.dot(Q,Q)/N)
    rms=float((np.dot(envelope,envelope)/N)**0.5)
    pow_db=float(10.*np.log10(np.dot(envelope,envelope)/N+1e-12))
    iq_c=_pearson(I,Q) if amp_std>1e-12 else 0.
    out[8:14]=[pow_db,iq_c,I_pow,Q_pow,I_pow/(Q_pow+1e-12),iq_c**2]
    nperseg=min(512,N//4)
    fw,psd=welch(envelope,fs=fs,nperseg=nperseg,noverlap=nperseg//2,return_onesided=True)
    pa=np.clip(np.abs(psd),1e-12,None); pa_sum=pa.sum()
    pd_db=10.*np.log10(pa); pk=int(pa.argmax())
    above=fw[pd_db>pd_db[pk]-10.]; bw_val=float(above.max()-above.min()) if len(above)>1 else 0.
    pn=pa/pa_sum; entropy=float(-np.dot(pn,np.log2(pn+1e-12)))
    cen=float(np.dot(fw,pa)/pa_sum); spread=float(np.sqrt(np.dot((fw-cen)**2,pa)/pa_sum))
    cs=np.cumsum(pa); rol=min(int(np.searchsorted(cs,0.85*cs[-1])),len(fw)-1)
    out[14:22]=[fw[pk],bw_val,entropy,cen,spread,fw[rol],float(pd_db.mean()),float(pd_db.max())]
    ifreq=np.diff(np.unwrap(np.angle(analytic)))
    if len(ifreq)>=2 and ifreq.std()>1e-8:
        out[22:26]=[float(ifreq.mean()),float(ifreq.std()),
                    float(ifreq.max()-ifreq.min()),float(kurtosis(ifreq))]
    else: out[22:26]=[0.]*4
    q_sz=max(1,len(pa)//4)
    b1=pa[:q_sz].sum()/pa_sum; b2=pa[q_sz:2*q_sz].sum()/pa_sum
    b3=pa[2*q_sz:3*q_sz].sum()/pa_sum; b4=pa[3*q_sz:].sum()/pa_sum
    out[26:30]=[b1,b2,b3,b4]
    stft_np=min(128,N//4)
    _,_,Zxx=stft(envelope,fs=fs,nperseg=stft_np,noverlap=stft_np//2,return_onesided=True)
    Sxx=np.abs(Zxx)**2+1e-12; fm=Sxx.mean(0); out[30]=float(np.diff(fm).var())
    bsz=max(1,Sxx.shape[0]//4)
    for b in range(4): out[31+b]=float(Sxx[b*bsz:(b+1)*bsz,:].mean(0).var())
    pa_s=np.sort(pa); L2=pa_s[1::2].mean()-pa_s[::2].mean()
    L4=(pa_s[3::4].mean()-3*pa_s[2::4].mean()+3*pa_s[1::4].mean()-pa_s[::4].mean())
    Sxx_n=Sxx.mean(1); Sxx_n/=Sxx_n.sum()+1e-12
    out[35:40]=[float(kurtosis(pa)),float(skew(pa)),float(L4/(L2+1e-12)),
                float(np.exp(np.log(pa+1e-12).mean()-np.log(pa.mean()+1e-12))),
                float(-np.dot(Sxx_n,np.log2(Sxx_n+1e-12)))]
    out[40:44]=[float((envelope.max()-envelope.min())/(amp_mean+1e-12)),
                float(envelope.max()/(rms+1e-12)),
                float(np.diff(ifreq).std()) if len(ifreq)>=2 else 0.,
                float((pa[fw>=cen].sum()-pa[fw<cen].sum())/(pa_sum+1e-12))]
    if len(envelope)>=4:
        acf=np.correlate(envelope-envelope.mean(),envelope-envelope.mean(),mode="full")
        acf=acf[len(acf)//2:]/(acf[len(acf)//2]+1e-12)
        acf_s=float(acf[min(10,len(acf)-1)]); acf_l=float(acf[min(200,len(acf)-1)])
        out[44:48]=[acf_s,float(acf[min(50,len(acf)-1)]),acf_l,float(acf_s/(acf_l+1e-12))]
    else: out[44:48]=[0.]*4
    out[48]=float(amp_kurt*entropy)
    out[49]=float(10.*np.log10((pa.max()/(pa.mean()+1e-12))+1e-12))
    out[50]=float(np.var(pa)); out[51]=float(kurtosis(envelope)) if amp_std>1e-8 else 0.
    out[52]=float((b3+b4)/(b1+b2+1e-9))
    return out

def safe_extract_rf(seg):
    try: return extract_rf_features(seg)
    except: return np.zeros(N_RF, dtype=np.float32)

def fuse_features(rf, flight=None, comm=None):
    fl=(np.asarray(flight,dtype=np.float32) if flight is not None else np.zeros(N_FLIGHT,np.float32))
    co=(np.asarray(comm,dtype=np.float32) if comm is not None else np.zeros(N_COMM,np.float32))
    return np.concatenate([rf.astype(np.float32),fl,co])


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 4b · 1D-CNN
# ─────────────────────────────────────────────────────────────────────────────
class CNN1D(nn.Module if TORCH_OK else object):
    def __init__(self, in_features, n_classes, embed_dim=CNN_EMBED_DIM, dropout=CNN_DROPOUT):
        if not TORCH_OK: return
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=7, padding=3), nn.ReLU(), nn.BatchNorm1d(32),
            nn.Conv1d(32, 64, kernel_size=5, padding=2), nn.ReLU(), nn.BatchNorm1d(64),
            nn.Dropout(dropout),
            nn.Conv1d(64, 64, kernel_size=3, padding=1), nn.ReLU(), nn.BatchNorm1d(64),
        )
        self.pool  = nn.AdaptiveAvgPool1d(1)
        self.embed = nn.Sequential(nn.Linear(64, embed_dim), nn.ReLU(), nn.Dropout(dropout/2))
        self.head  = nn.Linear(embed_dim, n_classes)

    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.conv(x)
        x = self.pool(x).squeeze(-1)
        e = self.embed(x)
        return self.head(e), e


class CNNExtractor:
    def __init__(self, n_classes):
        self.n_classes = n_classes; self.model = None
        self.scaler = RobustScaler(); self.fitted = False
        self.embed_dim = CNN_EMBED_DIM if TORCH_OK else 0
        self._trt_engine = None

    def fit(self, X, y):
        if not TORCH_OK:
            print("  [A2] CNN skipped — PyTorch unavailable"); return self
        t0 = time.time()
        X_sc = np.nan_to_num(self.scaler.fit_transform(X), nan=0., posinf=0., neginf=0.)
        X_sc = (X_sc - X_sc.min(axis=0)) / (X_sc.max(axis=0) - X_sc.min(axis=0) + 1e-9)
        Xt = torch.tensor(X_sc, dtype=torch.float32)
        yt = torch.tensor(y,    dtype=torch.long)
        ds = TensorDataset(Xt, yt)
        dl = DataLoader(ds, batch_size=CNN_BATCH, shuffle=True, drop_last=True)
        self.model = CNN1D(X.shape[1], self.n_classes).to(DEVICE)
        opt = optim.Adam(self.model.parameters(), lr=CNN_LR, weight_decay=1e-4)
        sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CNN_EPOCHS)
        loss_fn = nn.CrossEntropyLoss()
        self.model.train()
        for ep in range(CNN_EPOCHS):
            total_loss = 0.; correct = 0.; nb = 0
            for xb, yb in dl:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                opt.zero_grad(); logits, _ = self.model(xb)
                loss = loss_fn(logits, yb); loss.backward(); opt.step()
                total_loss += loss.item()
                correct += (logits.argmax(1) == yb).sum().item(); nb += len(yb)
            sched.step()
            if not PRODUCTION_MODE and (ep+1) % 10 == 0:
                print(f"    CNN ep {ep+1:>3}/{CNN_EPOCHS}  loss={total_loss/len(dl):.4f}  acc={correct/nb:.4f}")
        self.fitted = True; self.model.eval()
        print(f"  ✓ CNN trained  ({time.time()-t0:.1f}s)  device={DEVICE}")
        if EXPORT_TENSORRT and CUDA_OK:
            self._export_tensorrt(X.shape[1])
        return self

    def _export_tensorrt(self, in_features: int):
        try:
            from torch2trt import torch2trt
            dummy = torch.ones(1, in_features, dtype=torch.float32).to(DEVICE)
            class _Wrapper(nn.Module):
                def __init__(self, net): super().__init__(); self.net = net
                def forward(self, x): logits, _ = self.net(x); return logits
            wrapper = _Wrapper(self.model).eval()
            trt_model = torch2trt(wrapper, [dummy], int8_mode=True, max_batch_size=256)
            self._trt_engine = trt_model
            print("  ✓ [FIX-5] TensorRT INT8 engine ready")
        except ImportError:
            print("  ⚠  torch2trt not installed — using standard CUDA inference")
        except Exception as e:
            print(f"  ⚠  TensorRT export failed: {e}")

    def transform(self, X):
        if not self.fitted or self.model is None:
            return np.zeros((len(X), self.embed_dim), dtype=np.float32)
        X_sc = np.nan_to_num(self.scaler.transform(X), nan=0., posinf=0., neginf=0.)
        Xt = torch.tensor(X_sc, dtype=torch.float32); embs = []
        self.model.eval()
        with torch.no_grad():
            for i in range(0, len(Xt), 256):
                xb = Xt[i:i+256].to(DEVICE)
                _, e = self.model(xb)
                embs.append(e.cpu().numpy())
        return np.concatenate(embs, axis=0)

    def predict_proba(self, X):
        if not self.fitted or self.model is None:
            return np.ones((len(X), self.n_classes), dtype=np.float32) / self.n_classes
        X_sc = np.nan_to_num(self.scaler.transform(X), nan=0., posinf=0., neginf=0.)
        Xt   = torch.tensor(X_sc, dtype=torch.float32); probs = []
        if self._trt_engine is not None:
            with torch.no_grad():
                for i in range(0, len(Xt), 256):
                    logits = self._trt_engine(Xt[i:i+256].to(DEVICE))
                    probs.append(torch.softmax(logits, dim=-1).cpu().numpy())
        else:
            self.model.eval()
            with torch.no_grad():
                for i in range(0, len(Xt), 256):
                    logits, _ = self.model(Xt[i:i+256].to(DEVICE))
                    probs.append(torch.softmax(logits, dim=-1).cpu().numpy())
        return np.concatenate(probs, axis=0)


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 4c · STACKING META-LEARNER
# ─────────────────────────────────────────────────────────────────────────────
class StackingMetaLearner:
    def __init__(self):
        self.meta = LogisticRegression(
            C=0.5, max_iter=1000, class_weight="balanced",
            random_state=42, n_jobs=-1
        )
        self.scaler  = RobustScaler()
        self.fitted  = False
        self.classes_ = None

    def _make_stack(self, rf_p, gbt_p, gbp_p):
        return np.concatenate([rf_p, gbt_p, gbp_p], axis=1)

    def fit(self, rf_probs, gbt_probs, gbp_probs, y):
        X_stack = self._make_stack(rf_probs, gbt_probs, gbp_probs)
        X_sc    = self.scaler.fit_transform(X_stack)
        self.meta.fit(X_sc, y)
        self.classes_ = self.meta.classes_
        yp  = self.meta.predict(X_sc)
        acc = accuracy_score(y, yp)
        f1  = f1_score(y, yp, average="macro", zero_division=0)
        print(f"  ✓ [FIX-4] StackingMeta  train_acc={acc:.4f}  F1={f1:.4f}")
        self.fitted = True
        return self

    def predict_proba(self, rf_p, gbt_p, gbp_p):
        row    = np.concatenate([rf_p, gbt_p, gbp_p]).reshape(1, -1)
        row_sc = self.scaler.transform(row)
        return self.meta.predict_proba(row_sc)[0]


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 4d · FEATURE ROUTE CACHE
# ─────────────────────────────────────────────────────────────────────────────
class _FeatureCache:
    def __init__(self, maxsize=ROUTE_CACHE_MAXSIZE):
        self._cache  = {}
        self._order  = []
        self.maxsize = maxsize
        self.hits    = 0
        self.misses  = 0

    def _key(self, fv):
        q = np.round(fv * 20).astype(np.int16)
        return hashlib.blake2b(q.tobytes(), digest_size=6).hexdigest()

    def get(self, fv):
        k = self._key(fv)
        if k in self._cache:
            self.hits += 1
            return self._cache[k]
        self.misses += 1
        return None

    def put(self, fv, routed):
        k = self._key(fv)
        if len(self._order) >= self.maxsize:
            oldest = self._order.pop(0)
            self._cache.pop(oldest, None)
        self._cache[k] = routed
        self._order.append(k)

    def clear(self):
        self._cache.clear()
        self._order.clear()
        self.hits = self.misses = 0


_ROUTE_CACHE = _FeatureCache(maxsize=ROUTE_CACHE_MAXSIZE)
_HASH_IDX: List[Optional[np.ndarray]] = [None]


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 4e · LIGHTGBM WRAPPER
# ─────────────────────────────────────────────────────────────────────────────
class LGBClassifier:
    def __init__(self, n_estimators=100, mode="rf", n_classes=3,
                 num_leaves=63, lr=0.1, min_data_leaf=3,
                 subsample=0.8, colsample=0.5, device=None):
        self.n_estimators  = n_estimators
        self.mode          = mode
        self.n_classes     = n_classes
        self.num_leaves    = num_leaves
        self.lr            = lr
        self.min_data_leaf = min_data_leaf
        self.subsample     = subsample
        self.colsample     = colsample
        self.device        = device or _LGB_DEVICE
        self.booster: Optional[lgb.Booster] = None
        self.classes_      = np.arange(n_classes)

    def _base_params(self):
        params = {
            "objective":        "multiclass",
            "num_class":        self.n_classes,
            "num_leaves":       self.num_leaves,
            "min_data_in_leaf": self.min_data_leaf,
            "feature_fraction": self.colsample,
            "bagging_fraction": self.subsample,
            "bagging_freq":     1,
            "verbose":          -1,
            "n_jobs":           -1,
            "seed":             RANDOM_SEED,
            "device_type":      self.device,
        }
        if self.mode == "rf":
            params["boosting"] = "rf"
            params["learning_rate"] = 1.0
        else:
            params["boosting"]      = "gbdt"
            params["learning_rate"] = self.lr
        return params

    def fit(self, X, y):
        train_data = lgb.Dataset(X, label=y, free_raw_data=False)
        params     = self._base_params()
        callbacks  = [lgb.log_evaluation(period=-1)]
        t0 = time.time()
        self.booster = lgb.train(
            params, train_data,
            num_boost_round=self.n_estimators,
            callbacks=callbacks,
        )
        yp    = self.predict(X).argmax(1)
        acc   = accuracy_score(y, yp)
        f1    = f1_score(y, yp, average="macro", zero_division=0)
        mode_str = "RF" if self.mode == "rf" else "GBT"
        print(f"  ✓ LGB-{mode_str} [{self.device}]  "
              f"acc={acc:.4f}  F1={f1:.4f}  ({time.time()-t0:.1f}s)")
        return self

    def predict(self, X) -> np.ndarray:
        raw = self.booster.predict(X)
        if raw.ndim == 1:
            p1 = raw.reshape(-1, 1)
            return np.concatenate([1 - p1, p1], axis=1)
        return raw

    def predict_proba(self, X) -> np.ndarray:
        return self.predict(X)

    @property
    def oob_score_(self):
        return float("nan")


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 4f · [NEW-1] ACTION CONTROLLER  (Layer 3 of the AI-Harness)
# ─────────────────────────────────────────────────────────────────────────────
# Architecture note:
#   Layer 1 (Perception)  — RF classification stack (CNN / LGB-RF / Deep SVDD)
#   Layer 2 (Reasoning)   — SoftFusionEngine + TemporalTracker
#   Layer 3 (Action)      — ActionController  ← you are here
#
# To deploy on real hardware:
#   1. Subclass ActionController and override trigger_defense().
#   2. Inside trigger_defense(), open a serial port / REST API / CAN bus
#      to your jammer hardware and send the appropriate command.
#   3. Pass your subclass instance to make_classify_fn() as `action_ctrl`.
#
# Example (stub):
#   class RealHardwareController(ActionController):
#       def trigger_defense(self, threat_label, emitter_id, soft_score):
#           requests.post("http://jammer-api/jam",
#                         json={"freq": "2.4GHz", "duration_ms": 500})
# ─────────────────────────────────────────────────────────────────────────────

DEFENSE_LOG_PATH = "defense_log.txt"
THREAT_LABELS    = {"POTENTIAL_THREAT", "CONFIRMED_THREAT"}


class ActionController:
    """
    Software-Defined "Jammer" interface.

    MockJammer behaviour (default):
      - Appends a timestamped line to defense_log.txt.
      - Prints a console alert.
      - Does NOT transmit any RF signal.

    Production swap:
      Subclass and override trigger_defense() with real hardware I/O.
    """

    def __init__(self, log_path: str = DEFENSE_LOG_PATH, enabled: bool = True):
        self.log_path  = log_path
        self.enabled   = enabled
        self._actions  = 0          # total actions fired this session
        self._last_ts: Dict[str, float] = {}   # cooldown per emitter_id
        self._cooldown_s = 2.0      # minimum seconds between repeated triggers

    # ── public interface ─────────────────────────────────────────────────────
    def trigger_defense(self, threat_label: str, emitter_id: str = "",
                         soft_score: float = 0.0) -> bool:
        """
        Called by classify_signal when label is in THREAT_LABELS.

        Returns True if the action was actually fired, False if suppressed
        (disabled or within cooldown window).
        """
        if not self.enabled:
            return False
        # Per-emitter cooldown — avoids log spam on repeated bursts
        now = time.time()
        if emitter_id and (now - self._last_ts.get(emitter_id, 0.)) < self._cooldown_s:
            return False
        if emitter_id:
            self._last_ts[emitter_id] = now

        self._actions += 1
        self._log_action(threat_label, emitter_id, soft_score, now)
        self._console_alert(threat_label, emitter_id, soft_score)
        audit("ACTION_TRIGGERED",
              threat_label=threat_label, emitter_id=emitter_id[:8],
              soft_score=round(soft_score, 4), action_n=self._actions)
        return True

    def reset(self):
        self._actions = 0
        self._last_ts.clear()

    @property
    def total_actions(self) -> int:
        return self._actions

    # ── internal helpers ─────────────────────────────────────────────────────
    def _log_action(self, threat_label: str, emitter_id: str,
                     soft_score: float, ts: float):
        line = (f"[{time.strftime('%Y-%m-%dT%H:%M:%S', time.gmtime(ts))}Z] "
                f"ACTION: Jamming triggered against {threat_label} "
                f"| emitter={emitter_id[:8] if emitter_id else 'unknown'} "
                f"| score={soft_score:.4f} "
                f"| action_n={self._actions}\n")
        with open(self.log_path, "a") as f:
            f.write(line)

    def _console_alert(self, threat_label: str, emitter_id: str, soft_score: float):
        eid_short = emitter_id[:8] if emitter_id else "unknown"
        print(f"  📡 SENT SIGNAL TO JAMMER: {threat_label} "
              f"[emitter={eid_short}  score={soft_score:.4f}]")


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 4g · [NEW-2] LIVE STREAM SIMULATOR  (Virtual SDR)
# ─────────────────────────────────────────────────────────────────────────────
# This class is your "Software-in-the-Loop" test bench.
#
# How it works:
#   1. A background thread drips one burst (CSV row) every STREAM_INTERVAL_MS
#      into LIVE_STREAM_DIR/ as a timestamped .npy file.
#   2. run_live_stream_demo() polls LIVE_STREAM_DIR in the main thread, reads
#      each file, runs classify_signal(), then deletes the file.
#   3. The result is identical to having a real SDR write bursts to that folder.
#
# Feeding real CSVs:
#   Pass csv_dir="path/to/DroneRF" and the simulator will use actual recordings
#   instead of synthetic bursts.
# ─────────────────────────────────────────────────────────────────────────────

class LiveStreamSimulator:
    """
    Virtual SDR transmitter.

    Produces one feature-vector (burst) per STREAM_INTERVAL_MS into
    LIVE_STREAM_DIR, mimicking a streaming radio antenna.
    """

    def __init__(self,
                 out_dir: str = LIVE_STREAM_DIR,
                 interval_ms: float = STREAM_INTERVAL_MS,
                 max_bursts: int = STREAM_MAX_BURSTS,
                 csv_dir: Optional[str] = None,
                 cls_override: Optional[int] = None):
        self.out_dir     = out_dir
        self.interval_s  = interval_ms / 1000.
        self.max_bursts  = max_bursts
        self.csv_dir     = csv_dir
        self.cls_override = cls_override
        self._thread: Optional[threading.Thread] = None
        self._stop_evt   = threading.Event()
        self._burst_count = 0

        os.makedirs(out_dir, exist_ok=True)
        # Purge stale files from a previous run
        for f in Path(out_dir).glob("burst_*.npy"):
            f.unlink(missing_ok=True)

    # ── public API ────────────────────────────────────────────────────────────
    def start(self):
        self._stop_evt.clear()
        self._thread = threading.Thread(target=self._emit_loop, daemon=True)
        self._thread.start()
        print(f"  [LiveStream] Simulator started → {self.out_dir}/  "
              f"({self.interval_s*1000:.0f}ms intervals, max={self.max_bursts} bursts)")

    def stop(self):
        self._stop_evt.set()
        if self._thread:
            self._thread.join(timeout=5.)
        print(f"  [LiveStream] Simulator stopped.  Bursts emitted: {self._burst_count}")

    def poll_next(self) -> Optional[np.ndarray]:
        """
        Returns the oldest unprocessed burst from LIVE_STREAM_DIR,
        or None if the folder is empty.
        """
        candidates = sorted(Path(self.out_dir).glob("burst_*.npy"))
        if not candidates:
            return None
        path = candidates[0]
        try:
            fv = np.load(str(path))
        except Exception:
            path.unlink(missing_ok=True)
            return None
        path.unlink(missing_ok=True)
        return fv

    @property
    def burst_count(self) -> int:
        return self._burst_count

    @property
    def is_done(self) -> bool:
        return self._stop_evt.is_set()

    # ── internal ──────────────────────────────────────────────────────────────
    def _emit_loop(self):
        rng = np.random.default_rng(RANDOM_SEED + 200)
        csv_rows = self._load_csv_rows()
        csv_idx  = 0

        while not self._stop_evt.is_set() and self._burst_count < self.max_bursts:
            if csv_rows is not None:
                fv = csv_rows[csv_idx % len(csv_rows)]
                csv_idx += 1
            else:
                cls = self.cls_override if self.cls_override is not None else int(rng.integers(0, 3))
                fv  = _generate_rf_burst(cls, rng, noise_scale=1.2)

            fname = Path(self.out_dir) / f"burst_{self._burst_count:06d}.npy"
            np.save(str(fname), fv)
            self._burst_count += 1
            time.sleep(self.interval_s)

        self._stop_evt.set()

    def _load_csv_rows(self) -> Optional[np.ndarray]:
        """Load feature vectors from real CSV files (DroneRF format)."""
        if not self.csv_dir:
            return None
        root = Path(self.csv_dir)
        if not root.exists():
            print(f"  [LiveStream] csv_dir not found: {self.csv_dir} → using synthetic")
            return None
        rows = []
        for fp in sorted(root.rglob("*.csv"))[:10]:   # cap at 10 files for demo
            try:
                raw = pd.read_csv(fp, header=None, dtype=np.float32).values.ravel()
                if len(raw) < WINDOW_SIZE:
                    continue
                for start in range(0, len(raw) - WINDOW_SIZE, STEP_SIZE):
                    seg = raw[start: start + WINDOW_SIZE]
                    fv  = fuse_features(safe_extract_rf(seg))
                    rows.append(fv)
                    if len(rows) >= self.max_bursts * 2:
                        break
            except Exception:
                continue
            if len(rows) >= self.max_bursts * 2:
                break
        if not rows:
            return None
        print(f"  [LiveStream] Loaded {len(rows)} windows from {self.csv_dir}")
        return np.array(rows, dtype=np.float32)


def run_live_stream_demo(classify_signal_fn,
                          csv_dir: Optional[str] = None,
                          cls_override: Optional[int] = None,
                          max_bursts: int = STREAM_MAX_BURSTS,
                          interval_ms: float = STREAM_INTERVAL_MS):
    """
    End-to-end live-stream integration test.

    Starts the Virtual SDR in a background thread, then polls LIVE_STREAM_DIR
    from the main thread and runs classify_signal_fn on each arriving burst,
    exactly as a real antenna would.

    Parameters
    ----------
    classify_signal_fn : callable
        The hysteresis-wrapped classify_signal from make_classify_fn().
    csv_dir : str, optional
        Path to DroneRF CSV directory.  None → use synthetic bursts.
    cls_override : int, optional
        Force a specific class (0=BG, 1=AR, 2=Phantom) for all bursts.
    """
    print(f"\n{'═'*65}")
    print("  [NEW-2] LIVE STREAM DEMO  (Virtual SDR / SITL)")
    print(f"  Interval={interval_ms}ms  MaxBursts={max_bursts}")
    print(f"{'═'*65}")

    sim = LiveStreamSimulator(
        out_dir=LIVE_STREAM_DIR,
        interval_ms=interval_ms,
        max_bursts=max_bursts,
        csv_dir=csv_dir,
        cls_override=cls_override,
    )
    sim.start()

    processed    = 0
    label_counts = Counter()
    latencies_ms = []

    try:
        while processed < max_bursts:
            fv = sim.poll_next()
            if fv is None:
                if sim.is_done and processed >= sim.burst_count:
                    break
                time.sleep(0.005)
                continue

            t0  = time.perf_counter()
            dec = classify_signal_fn(fv)
            lat = (time.perf_counter() - t0) * 1000

            label = dec.get("label", "UNKNOWN")
            label_counts[label] += 1
            latencies_ms.append(lat)
            processed += 1

            icon = DECISION_ICONS.get(label, "?")
            print(f"  Burst {processed:>3}  {icon} {label:<28} "
                  f"score={dec.get('soft_score', 0.):.3f}  lat={lat:.1f}ms")

    finally:
        sim.stop()

    if latencies_ms:
        arr = np.array(latencies_ms)
        print(f"\n  Processed {processed} bursts in live-stream mode")
        print(f"  Latency — mean={arr.mean():.1f}ms  p95={np.percentile(arr,95):.1f}ms")
        print(f"  Label distribution:")
        for lbl, cnt in label_counts.most_common():
            print(f"    {DECISION_ICONS.get(lbl,'?')} {lbl:<32} {cnt:>3}  ({cnt/processed:.0%})")

    # Clean up temp folder
    for f in Path(LIVE_STREAM_DIR).glob("burst_*.npy"):
        f.unlink(missing_ok=True)

    return label_counts, latencies_ms


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 5 · DATA PIPELINE
# ─────────────────────────────────────────────────────────────────────────────
def build_or_load_dataset(data_dir, output_csv=OUTPUT_CSV):
    cache = Path(output_csv)
    if cache.exists():
        try:
            df = pd.read_csv(output_csv)
            if ("high_low_band_ratio" in df.columns and
                    len([c for c in df.columns if c in RF_FEATURE_NAMES]) == N_RF and
                    df["amp_std"].var() > 1e-4 and
                    "synthetic" not in str(df["source_file"].iloc[0])):
                for col in ALL_FEATURE_NAMES:
                    if col not in df.columns: df[col] = 0.
                print(f"⚡ Cache loaded: {output_csv}  ({len(df):,} rows)")
                return df
            else:
                print("  Cache is synthetic or stale → rebuilding")
        except Exception as e:
            print(f"  Cache load failed: {e}")
        cache.unlink(missing_ok=True)

    if not (data_dir and Path(data_dir).exists()):
        print("  ⚠  DATA_DIR not found → synthetic fallback")
        df = generate_realistic_dataset()
        df.to_csv(output_csv, index=False)
        return df

    print(f"\nBuilding from real data: {data_dir} ...")
    root = Path(data_dir)
    folder_class = {}
    for subdir in sorted(root.iterdir()):
        if not subdir.is_dir(): continue
        name_lower = subdir.name.lower()
        cls = next((v for k, v in FOLDER_MAP.items() if k in name_lower), None)
        if cls is not None:
            folder_class[subdir] = cls
            print(f"  Folder '{subdir.name}' → class {cls} ({CLASS_NAMES[cls]})")

    if not folder_class:
        print("  ⚠  No folders matched → synthetic fallback")
        df = generate_realistic_dataset(); df.to_csv(output_csv, index=False); return df

    class_files = {}
    for folder, cls in folder_class.items():
        csv_files = sorted(folder.rglob("*.csv"))
        if csv_files:
            class_files.setdefault(cls, []).extend(csv_files)

    if not class_files:
        print("  ⚠  No CSVs found → synthetic fallback")
        df = generate_realistic_dataset(); df.to_csv(output_csv, index=False); return df

    rng      = np.random.default_rng(RANDOM_SEED)
    q        = TARGET_TOTAL // len(class_files)
    rows, labels, fnames = [], [], []
    for cls in sorted(class_files.keys()):
        flist = list(class_files[cls]); rng.shuffle(flist)
        count = 0; skipped = 0
        for fp in flist:
            if count >= q: break
            try:
                raw = pd.read_csv(fp, header=None, dtype=np.float32).values.ravel()
            except Exception:
                skipped += 1; continue
            if len(raw) < WINDOW_SIZE:
                skipped += 1; continue
            start = 0
            while start + WINDOW_SIZE <= len(raw) and count < q:
                seg = raw[start: start + WINDOW_SIZE]
                fv  = fuse_features(safe_extract_rf(seg))
                rows.append(fv); labels.append(cls); fnames.append(fp.name)
                start += STEP_SIZE; count += 1
        print(f"  ✓ Class {cls} ({CLASS_NAMES[cls]}): {count} windows (skipped {skipped})")

    if not rows:
        print("  ⚠  No windows extracted → synthetic fallback")
        df = generate_realistic_dataset(); df.to_csv(output_csv, index=False); return df

    X  = np.array(rows, dtype=np.float32)
    df = pd.DataFrame(X, columns=ALL_FEATURE_NAMES)
    df.insert(0, "label_int",  labels)
    df.insert(1, "label_name", [CLASS_NAMES[c] for c in labels])
    df.insert(2, "source_file", fnames)
    df = df.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
    df.to_csv(output_csv, index=False)
    return df


def prepare_data(df):
    X_all = np.nan_to_num(
        df[ALL_FEATURE_NAMES].fillna(0).values.astype(np.float32),
        nan=0., posinf=0., neginf=0.)
    y_all = df["label_int"].values.astype(np.int64)
    known = sorted([c for c in np.unique(y_all)
                    if c in CLASS_NAMES and (y_all == c).sum() >= 6])
    mask  = np.isin(y_all, known)
    X_use, y_use = X_all[mask], y_all[mask]
    lmap  = {old: new for new, old in enumerate(known)}
    y_map = np.array([lmap[yi] for yi in y_use], dtype=np.int64)
    CP    = [CLASS_NAMES[c] for c in known]
    print(f"\n  Training classes: {len(CP)}")
    for i, cn in enumerate(CP):
        print(f"    [{i}] {cn:<20} ({(y_map == i).sum()} windows)")
    return X_use, y_map, lmap, CP, len(CP)


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 6 · FEATURE ROUTER
# ─────────────────────────────────────────────────────────────────────────────
class FeatureRouter:
    def __init__(self, rf_idx, gbt_idx, master_idx,
                 scaler_rf, scaler_gbt, scaler_master, scaler_sub, sub_idx):
        self.rf_idx=rf_idx; self.gbt_idx=gbt_idx
        self.master_idx=master_idx; self.sub_idx=sub_idx
        self.scaler_rf=scaler_rf; self.scaler_gbt=scaler_gbt
        self.scaler_master=scaler_master; self.scaler_sub=scaler_sub

    def route(self, fv_raw):
        X = fv_raw if fv_raw.ndim==2 else fv_raw.reshape(1,-1)
        X = np.nan_to_num(X.astype(np.float32), nan=0., posinf=0., neginf=0.)
        def _s(sc,idx):
            return np.nan_to_num(sc.transform(X[:,idx]), nan=0., posinf=0., neginf=0.)
        return {"rf":_s(self.scaler_rf,self.rf_idx),
                "gbt":_s(self.scaler_gbt,self.gbt_idx),
                "master":_s(self.scaler_master,self.master_idx),
                "sub":_s(self.scaler_sub,self.sub_idx)}


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 7 · FEATURE SELECTION
# ─────────────────────────────────────────────────────────────────────────────
def validate_and_select_features(X, y):
    print(f"\n{'='*60}\nFEATURE SELECTION\n{'='*60}")
    sc_pre = RobustScaler()
    X_s    = np.nan_to_num(sc_pre.fit_transform(X), nan=0., posinf=0., neginf=0.)
    mi       = mutual_info_classif(X_s, y, random_state=RANDOM_SEED)
    top_mi   = np.argsort(mi)[::-1]
    top_var  = np.argsort(X_s.var(0))[::-1]
    rf_idx   = top_mi[:RF_TOP_K_MI]
    gbt_idx  = top_var[:GBT_TOP_K_VAR]
    master_idx = top_mi
    sub_names = [f for f in SUBCLF_FEATURES if f in FEAT_IDX]
    sub_idx   = np.array([FEAT_IDX[f] for f in sub_names], dtype=np.int64)
    print(f"  RF(MI-top-{RF_TOP_K_MI}) | GBT(Var-top-{GBT_TOP_K_VAR})")

    def _fs(idx):
        sc=RobustScaler()
        Xs=np.nan_to_num(sc.fit_transform(X[:,idx]),nan=0.,posinf=0.,neginf=0.)
        return sc, Xs

    scaler_rf,X_rf         = _fs(rf_idx)
    scaler_gbt,X_gbt       = _fs(gbt_idx)
    scaler_master,X_master = _fs(master_idx)
    scaler_sub,X_sub       = _fs(sub_idx)
    router = FeatureRouter(rf_idx, gbt_idx, master_idx,
                            scaler_rf, scaler_gbt, scaler_master, scaler_sub, sub_idx)
    return router, mi, X_master, X_rf, X_gbt, X_sub


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 8 · CLASSIFIERS
# ─────────────────────────────────────────────────────────────────────────────
class GaussianBayesPosterior:
    def __init__(self, temperature=GBP_TEMPERATURE, var_smoothing=1e-3):
        self.tau=temperature; self.vsf=var_smoothing; self.fitted=False

    def fit(self, X, y):
        classes=np.unique(y); self.classes_=classes
        smooth=self.vsf*X.var(0).mean()
        self.mu_={}; self.var_={}; self.log_prior_={}
        for k in classes:
            Xk=X[y==k]
            self.mu_[k]=Xk.mean(0); self.var_[k]=Xk.var(0)+smooth
            self.log_prior_[k]=float(np.log(len(Xk)/len(y)))
        self.fitted=True; print(f"  ✓ GBP  τ={self.tau}"); return self

    def predict_proba(self, X):
        X=np.asarray(X,dtype=np.float64)
        lp=np.stack([-0.5*((X-self.mu_[k])**2/self.var_[k]).sum(1)/self.tau
                     -0.5*np.log(2*np.pi*self.var_[k]).sum()/self.tau
                     +self.log_prior_[k] for k in self.classes_],axis=1)
        lp-=lp.max(1,keepdims=True); p=np.exp(lp); p/=p.sum(1,keepdims=True)
        return p

    def predict(self, X): return self.predict_proba(X).argmax(1)


class EnsembleUncertainty:
    def __init__(self, n_models=N_ENSEMBLE_TREES, subsample=ENSEMBLE_SUBSAMPLE):
        self.n_models=n_models; self.subsample=subsample; self.models=[]

    def fit(self, X, y):
        n_cls = len(np.unique(y))
        print(f"  [Ensemble] Training {self.n_models} bootstrap sub-models "
              f"({'LGB' if LGB_OK else 'RF'}) ...")
        rng=np.random.default_rng(RANDOM_SEED); n=len(X)
        for i in range(self.n_models):
            idx=rng.choice(n,size=int(n*self.subsample),replace=True)
            if LGB_OK:
                m = LGBClassifier(n_estimators=200, mode="rf", n_classes=n_cls,
                                   num_leaves=31, min_data_leaf=3,
                                   subsample=0.8, colsample=0.5)
            else:
                from sklearn.ensemble import RandomForestClassifier as RFC
                m = RFC(200, max_features="sqrt", min_samples_leaf=3,
                        class_weight="balanced",
                        random_state=int(rng.integers(0,99999)), n_jobs=-1)
            m.fit(X[idx], y[idx]); self.models.append(m)
        avg_p = np.mean([m.predict_proba(X) for m in self.models], axis=0)
        f1 = f1_score(y, avg_p.argmax(1), average="macro", zero_division=0)
        print(f"  ✓ Ensemble F1 (train)={f1:.4f}"); return self

    def predict_with_uncertainty(self, X):
        probs=np.stack([m.predict_proba(X) for m in self.models],axis=0)
        mean_p=probs.mean(0); epistemic=probs.var(0).sum(-1)
        aleatoric=-(mean_p*np.log(mean_p+1e-12)).sum(-1)
        return mean_p, epistemic, aleatoric


class PhantomARSubClassifier:
    def __init__(self): self.model=None; self.fitted=False

    def fit(self, X_sub, y):
        mask=np.isin(y,[1,2])
        if mask.sum()<20: return self
        Xs=X_sub[mask]; ys=(y[mask]==2).astype(np.int64)
        if LGB_OK:
            self.model = LGBClassifier(n_estimators=300, mode="gbdt", n_classes=2,
                                        num_leaves=15, lr=0.05, min_data_leaf=3,
                                        subsample=0.8, colsample=0.7)
        else:
            from sklearn.ensemble import GradientBoostingClassifier as GBC
            self.model = GBC(n_estimators=300, learning_rate=0.05, max_depth=4,
                             subsample=0.8, min_samples_leaf=3, random_state=RANDOM_SEED)
        self.model.fit(Xs,ys)
        yp  = self.model.predict_proba(Xs).argmax(1)
        f1  = f1_score(ys, yp, average="binary", zero_division=0)
        print(f"  ✓ PhantomARSubClassifier  train_F1={f1:.4f}")
        self.fitted=True; return self

    def p_phantom(self, X_sub):
        if not self.fitted or self.model is None: return 0.5
        return float(self.model.predict_proba(X_sub)[0,1])


class TemperatureScaler:
    def __init__(self): self.T=1.0; self._ece=None

    def fit(self, logits, y):
        def ece_fn(T):
            T=max(T,TEMP_MIN); s=logits/T
            e=np.exp(s-s.max(1,keepdims=True)); p=e/e.sum(1,keepdims=True)
            pred=p.argmax(1); acc=(pred==y).astype(float); conf=p.max(1)
            return float(np.mean((conf-acc)**2))
        res=minimize_scalar(ece_fn,bounds=(TEMP_MIN,TEMP_MAX),method="bounded")
        self.T=float(np.clip(res.x,TEMP_MIN,TEMP_MAX)); self._ece=ece_fn(self.T)
        print(f"  ✓ TemperatureScaler  T={self.T:.4f}  ECE={self._ece:.4f}"); return self

    def calibrate(self, logits):
        T=max(self.T,TEMP_MIN); s=logits/T
        e=np.exp(s-s.max(1,keepdims=True)); return e/e.sum(1,keepdims=True)

    def expected_calibration_error(self, probs, y, n_bins=10):
        confs=probs.max(1); preds=probs.argmax(1); acc=(preds==y).astype(float); ece=0.
        for b in range(n_bins):
            lo,hi=b/n_bins,(b+1)/n_bins; mask=(confs>=lo)&(confs<hi)
            if mask.sum()==0: continue
            ece+=mask.sum()/len(y)*abs(acc[mask].mean()-confs[mask].mean())
        return float(ece)


class LaplaceApproximation:
    def __init__(self,precision=LAPLACE_PRIOR_PRECISION,n_samples=LAPLACE_N_SAMPLES):
        self.alpha=precision; self.n_samples=n_samples; self.fitted=False

    def fit(self,lr_model,X,y,n_classes):
        t0=time.time(); self.n_classes=n_classes; D=X.shape[1]
        self.W_map=lr_model.coef_.astype(np.float64)
        self.b_map=lr_model.intercept_.astype(np.float64)
        Z=X@self.W_map.T+self.b_map; Z-=Z.max(1,keepdims=True)
        eZ=np.exp(Z); probs=eZ/eZ.sum(1,keepdims=True)
        self.chol_factors=[]
        for k in range(n_classes):
            pi=probs[:,k].clip(1e-7,1-1e-7); w=pi*(1-pi)
            H=(X*w[:,None]).T@X+self.alpha*np.eye(D)
            try: self.chol_factors.append(("chol",cho_factor(H,lower=False,check_finite=False),H))
            except: self.chol_factors.append(("pinv",np.linalg.pinv(H),H))
        self.fitted=True; print(f"  ✓ Laplace  ({time.time()-t0:.2f}s)"); return self

    def predictive_variance(self,X):
        if not self.fitted or PRODUCTION_MODE: return 0.
        X=np.asarray(X,dtype=np.float64); C=self.n_classes
        samples=np.zeros((self.n_samples,X.shape[0],C))
        for k in range(C):
            kind,factor,H=self.chol_factors[k]; D=self.W_map.shape[1]
            z=np.random.randn(self.n_samples,D)
            if kind=="chol":
                try: v=cho_solve(factor,z.T,check_finite=False).T
                except: v=z/(np.diag(H)+1e-8)
            else:
                try: v=(np.linalg.cholesky(factor+1e-8*np.eye(D))@z.T).T
                except: v=z*np.sqrt(np.diag(factor)+1e-8)
            samples[:,:,k]=(X@(self.W_map[k]+v).T+self.b_map[k]).T
        Z=samples-samples.max(-1,keepdims=True); p=np.exp(Z); p/=p.sum(-1,keepdims=True)
        return float(p.var(0).mean())


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 8b · LEGACY OPEN-SET DETECTOR
# ─────────────────────────────────────────────────────────────────────────────
class _LegacyOpenSetDetector:
    def __init__(self,nu=OCSVM_NU,gamma=OCSVM_GAMMA,n_pca=12):
        self.nu=nu; self.gamma=gamma; self.n_pca=n_pca
        self.models={}; self.pca=None; self.fitted=False
        self._lo={}; self._hi={}

    def fit(self,X_master,y):
        t0=time.time()
        n_comp=min(self.n_pca,X_master.shape[1],X_master.shape[0]-1)
        self.pca=PCA(n_components=n_comp,random_state=RANDOM_SEED)
        X_pca=self.pca.fit_transform(X_master)
        for k in np.unique(y):
            Xk=X_pca[y==k]
            m=OneClassSVM(nu=self.nu,kernel="rbf",gamma=self.gamma); m.fit(Xk)
            self.models[k]=m
            scores=m.decision_function(Xk)
            self._lo[k]=float(np.percentile(scores,1)); self._hi[k]=float(np.percentile(scores,99))
            if self._hi[k]<=self._lo[k]: self._hi[k]=self._lo[k]+1.
        self.fitted=True; print(f"  ✓ LegacyOSD  ({time.time()-t0:.2f}s)"); return self

    def inclusion_score(self,X_master):
        X_pca=self.pca.transform(np.asarray(X_master,dtype=np.float64))
        scores=[]
        for k,m in self.models.items():
            raw=m.decision_function(X_pca)
            norm=np.clip((raw-self._lo[k])/(self._hi[k]-self._lo[k]+1e-9),0.,1.)
            scores.append(norm)
        return np.stack(scores,axis=1).max(1)


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 8c · DEEP SVDD
# ─────────────────────────────────────────────────────────────────────────────
class _SVDDNet(nn.Module if TORCH_OK else object):
    def __init__(self, in_dim, embed_dim=SVDD_EMBED_DIM):
        if not TORCH_OK: return
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 128), nn.BatchNorm1d(128), nn.LeakyReLU(0.1),
            nn.Linear(128, 64),     nn.BatchNorm1d(64),  nn.LeakyReLU(0.1),
            nn.Linear(64, embed_dim, bias=False),
        )
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.net(x)


class DeepSVDDDetector:
    def __init__(self, nu=SVDD_NU, embed_dim=SVDD_EMBED_DIM):
        self.nu = nu; self.embed_dim = embed_dim
        self.net = None; self.centre = None; self.radius = None
        self.scaler = RobustScaler(); self.fitted = False
        self._fallback = _LegacyOpenSetDetector()

    def fit(self, X_master, y):
        if not TORCH_OK:
            self._fallback.fit(X_master, y)
            self.fitted = True
            return self

        t0 = time.time()
        in_dim = X_master.shape[1]
        X_sc = np.nan_to_num(self.scaler.fit_transform(X_master), nan=0., posinf=0., neginf=0.)
        Xt = torch.tensor(X_sc, dtype=torch.float32)
        self.net = _SVDDNet(in_dim, self.embed_dim).to(DEVICE)

        self.net.eval()
        with torch.no_grad():
            embs = nn.functional.normalize(self.net(Xt.to(DEVICE)), p=2, dim=1)
            c = embs.mean(0)
            self.centre = nn.functional.normalize(
                c.unsqueeze(0), p=2, dim=1).squeeze(0).detach()

        opt = torch.optim.Adam(filter(lambda p: p.requires_grad, self.net.parameters()),
                               lr=SVDD_LR, weight_decay=1e-5)
        ds = torch.utils.data.TensorDataset(Xt)
        dl = torch.utils.data.DataLoader(ds, batch_size=SVDD_BATCH, shuffle=True)

        self.net.train()
        for ep in range(SVDD_EPOCHS):
            for (xb,) in dl:
                xb = xb.to(DEVICE)
                opt.zero_grad()
                emb = nn.functional.normalize(self.net(xb), p=2, dim=1)
                dist = ((emb - self.centre) ** 2).sum(dim=1)
                loss = torch.mean(dist)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.net.parameters(), max_norm=1.0)
                opt.step()

        self.net.eval()
        with torch.no_grad():
            all_embs = []
            for i in range(0, len(Xt), 256):
                e = nn.functional.normalize(self.net(Xt[i:i+256].to(DEVICE)), p=2, dim=1)
                all_embs.append(e.cpu())
            all_embs = torch.cat(all_embs)
            dists = ((all_embs - self.centre.cpu()) ** 2).sum(dim=1).sqrt()
            self.radius = float(torch.quantile(dists, 1.0 - self.nu).item()) + 1e-6

        self.fitted = True
        print(f"  ✓ [P1] DeepSVDD  Radius={self.radius:.4f}  "
              f"device={DEVICE}  ({time.time()-t0:.1f}s)")
        return self

    def _raw_distances(self, X_master):
        if not TORCH_OK or self.net is None:
            return 1. - self._fallback.inclusion_score(X_master)
        X_sc = np.nan_to_num(
            self.scaler.transform(np.asarray(X_master, dtype=np.float32)),
            nan=0., posinf=0., neginf=0.)
        Xt = torch.tensor(X_sc, dtype=torch.float32)
        self.net.eval()
        dists = []
        with torch.no_grad():
            for i in range(0, len(Xt), 256):
                emb = nn.functional.normalize(self.net(Xt[i:i+256].to(DEVICE)), p=2, dim=1)
                d   = ((emb - self.centre) ** 2).sum(dim=1).sqrt()
                dists.append(d.cpu().numpy())
        return np.concatenate(dists)

    def inclusion_score(self, X_master):
        if not self.fitted:
            return np.ones(len(X_master), dtype=np.float32) * 0.5
        if not TORCH_OK:
            return self._fallback.inclusion_score(X_master)
        raw = self._raw_distances(X_master)
        return np.clip(1.0 - (raw / (self.radius * 2.0)), 0., 1.)


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 9 · ANOMALY DETECTORS
# ─────────────────────────────────────────────────────────────────────────────
class MahalanobisDetector:
    def fit(self,X_master,y):
        self.params={}
        for c in np.unique(y):
            Xc=X_master[y==c]; mu=Xc.mean(0)
            cov=np.cov(Xc,rowvar=False)+np.eye(Xc.shape[1])*1e-2
            try: prec=np.linalg.inv(cov)
            except: prec=np.linalg.pinv(cov)
            self.params[c]=(mu,prec)
        raw=self.score(X_master)
        self._lo=float(np.percentile(raw,1)); self._hi=float(np.percentile(raw,99))
        if self._hi<=self._lo: self._hi=self._lo+1.
        self.threshold=float(np.percentile(raw,99))
        return self

    def score(self,X):
        dists=[]
        for mu,prec in self.params.values():
            d=X-mu
            dists.append(np.sqrt(np.maximum(np.einsum("ni,ij,nj->n",d,prec,d),0.)))
        return np.nan_to_num(np.stack(dists,1).min(1),nan=0.,posinf=0.,neginf=0.)

    def norm_score(self,X):
        return np.clip((self.score(X)-self._lo)/(self._hi-self._lo+1e-9),0.,1.)


class IsoForestDetector:
    def fit(self,X_master,y=None):
        self.model=IsolationForest(n_estimators=ISO_N_ESTIMATORS,
            contamination=ISO_CONTAMINATION,n_jobs=-1,random_state=RANDOM_SEED)
        self.model.fit(X_master)
        raw=self.score(X_master)
        self._lo=float(np.percentile(raw,1)); self._hi=float(np.percentile(raw,99))
        if self._hi<=self._lo: self._hi=self._lo+1.
        return self

    def score(self,X):
        return np.nan_to_num(-self.model.score_samples(X),nan=0.,posinf=0.,neginf=0.)

    def norm_score(self,X):
        return np.clip((self.score(X)-self._lo)/(self._hi-self._lo+1e-9),0.,1.)


class ThreatScorer:
    def __init__(self,dm,di,X_master_train):
        self.dm=dm; self.di=di; self.wm=ANOMALY_W_MAHAL; self.wi=ANOMALY_W_ISO
        self.cap=ANOMALY_SCORE_CAP
        raw_thr=float(np.percentile(self.compute_raw(X_master_train),97))
        self.threshold=max(raw_thr,0.72)
        print(f"  Threat: mahal={self.wm}  isoforest={self.wi}  threshold={self.threshold:.4f}")

    def compute_raw(self,X_master):
        return self.wm*self.dm.norm_score(X_master)+self.wi*self.di.norm_score(X_master)

    def compute(self,X_master):
        return np.minimum(self.compute_raw(X_master),self.cap)


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 10 · BUILD & EVALUATE
# ─────────────────────────────────────────────────────────────────────────────
def build_and_evaluate(router, X_raw_full, y, X_master, X_rf, X_gbt, X_sub, classes_present):
    print(f"\n{'='*60}\nMODEL TRAINING  (v32-FIELD)\n{'='*60}")
    print(f"  Classifier backend: {'LightGBM' if LGB_OK else 'scikit-learn'} "
          f"| device={_LGB_DEVICE}")
    rng_aug = np.random.default_rng(RANDOM_SEED + 1)
    N_CLS   = len(classes_present)

    idx_tr, idx_te = train_test_split(
        np.arange(len(y)), test_size=0.20, stratify=y, random_state=RANDOM_SEED)

    X_tr_raw = X_raw_full[idx_tr]; y_tr = y[idx_tr]
    X_te_raw = X_raw_full[idx_te]; y_te = y[idx_te]

    X_tr_aug, y_tr_aug = mixup_augment(X_tr_raw, y_tr, rng_aug)

    def _scale_aug(X_aug, sc, idx):
        return np.nan_to_num(sc.transform(X_aug[:,idx]), nan=0., posinf=0., neginf=0.)

    X_m_aug  = _scale_aug(X_tr_aug, router.scaler_master, router.master_idx)
    X_rf_aug = _scale_aug(X_tr_aug, router.scaler_rf,     router.rf_idx)
    X_gb_aug = _scale_aug(X_tr_aug, router.scaler_gbt,    router.gbt_idx)
    X_sb_aug = _scale_aug(X_tr_aug, router.scaler_sub,    router.sub_idx)

    X_te_rf  = X_rf[idx_te];  X_te_gbt = X_gbt[idx_te]
    X_te_sub = X_sub[idx_te]; X_te_m   = X_master[idx_te]

    _, cnts = np.unique(y_tr_aug, return_counts=True)
    k_sm = max(1, min(5, int(cnts.min()) - 1))
    def _smote(X, y_): return SMOTE(random_state=RANDOM_SEED, k_neighbors=k_sm).fit_resample(X, y_)
    X_sm_m,  y_sm_m  = _smote(X_m_aug,  y_tr_aug)
    X_sm_rf, y_sm_rf = _smote(X_rf_aug, y_tr_aug)
    X_sm_gb, y_sm_gb = _smote(X_gb_aug, y_tr_aug)
    X_sm_sb, y_sm_sb = _smote(X_sb_aug, y_tr_aug)
    print(f"  SMOTE: master={X_sm_m.shape[0]:,}  RF={X_sm_rf.shape[0]:,}  GBT={X_sm_gb.shape[0]:,}")

    print(f"\n  [A2] Training 1D-CNN  (device={DEVICE}) ...")
    cnn = CNNExtractor(n_classes=N_CLS)
    cnn.fit(X_tr_aug, y_tr_aug)

    print(f"\n  [A] Training LGB-RF ...")
    if LGB_OK:
        rf = LGBClassifier(n_estimators=LGB_RF_N_ESTIMATORS, mode="rf",
                            n_classes=N_CLS, num_leaves=LGB_RF_NUM_LEAVES,
                            min_data_leaf=LGB_RF_MIN_DATA_LEAF,
                            subsample=LGB_RF_SUBSAMPLE, colsample=LGB_RF_COLSAMPLE)
        rf.fit(X_sm_rf, y_sm_rf)
    else:
        from sklearn.ensemble import RandomForestClassifier as RFC
        rf = RFC(500, class_weight="balanced", max_features="sqrt", min_samples_leaf=3,
                 random_state=RANDOM_SEED, n_jobs=-1, oob_score=True)
        rf.fit(X_sm_rf, y_sm_rf)

    yp_rf  = rf.predict_proba(X_te_rf).argmax(1)
    acc_rf = accuracy_score(y_te, yp_rf)
    f1_rf  = f1_score(y_te, yp_rf, average="macro", zero_division=0)
    print(f"  [A] RF test  acc={acc_rf:.4f}  F1={f1_rf:.4f}")

    print(f"\n  [A1] Hard-negative mining ...")
    X_tr_hn, y_tr_hn = hard_negative_mine(
        X_tr_aug, y_tr_aug, rf, router.scaler_rf, router.rf_idx, rng_aug)
    if len(X_tr_hn) > len(X_tr_aug):
        X_hn_rf = _scale_aug(X_tr_hn, router.scaler_rf,     router.rf_idx)
        X_sm_rf2, y_sm_rf2 = _smote(X_hn_rf, y_tr_hn)
        if LGB_OK:
            rf = LGBClassifier(n_estimators=LGB_RF_N_ESTIMATORS, mode="rf",
                                n_classes=N_CLS, num_leaves=LGB_RF_NUM_LEAVES,
                                min_data_leaf=LGB_RF_MIN_DATA_LEAF,
                                subsample=LGB_RF_SUBSAMPLE, colsample=LGB_RF_COLSAMPLE)
        else:
            rf = RFC(500, class_weight="balanced", max_features="sqrt", min_samples_leaf=3,
                     random_state=RANDOM_SEED, n_jobs=-1, oob_score=True)
        rf.fit(X_sm_rf2, y_sm_rf2)
        yp_rf  = rf.predict_proba(X_te_rf).argmax(1)
        acc_rf = accuracy_score(y_te, yp_rf)
        f1_rf  = f1_score(y_te, yp_rf, average="macro", zero_division=0)
        print(f"  [A] RF (post-HNM) acc={acc_rf:.4f}  F1={f1_rf:.4f}")
        X_hn_gb = _scale_aug(X_tr_hn, router.scaler_gbt,    router.gbt_idx)
        X_hn_m  = _scale_aug(X_tr_hn, router.scaler_master, router.master_idx)
        X_sm_gb, y_sm_gb = _smote(X_hn_gb, y_tr_hn)
        X_sm_m,  y_sm_m  = _smote(X_hn_m,  y_tr_hn)

    print(f"\n  [B] Training LGB-GBT ...")
    if LGB_OK:
        gbt = LGBClassifier(n_estimators=LGB_GBT_N_ESTIMATORS, mode="gbdt",
                             n_classes=N_CLS, num_leaves=LGB_GBT_NUM_LEAVES,
                             lr=LGB_GBT_LR, min_data_leaf=LGB_GBT_MIN_DATA_LEAF,
                             subsample=LGB_GBT_SUBSAMPLE)
        gbt.fit(X_sm_gb, y_sm_gb)
    else:
        from sklearn.ensemble import GradientBoostingClassifier as GBC
        gbt = GBC(n_estimators=200, learning_rate=0.08, max_depth=5,
                  subsample=0.8, min_samples_leaf=5, random_state=RANDOM_SEED)
        gbt.fit(X_sm_gb, y_sm_gb)

    yp_gbt  = gbt.predict_proba(X_te_gbt).argmax(1)
    acc_gbt = accuracy_score(y_te, yp_gbt)
    f1_gbt  = f1_score(y_te, yp_gbt, average="macro", zero_division=0)
    print(f"  [B] GBT test  acc={acc_gbt:.4f}  F1={f1_gbt:.4f}")

    lr_clf = LogisticRegression(C=1.0, class_weight="balanced", max_iter=1000,
             random_state=RANDOM_SEED, n_jobs=-1)
    lr_clf.fit(X_sm_m, y_sm_m)
    yp_lr  = lr_clf.predict(X_te_m)
    acc_lr = accuracy_score(y_te, yp_lr)
    f1_lr  = f1_score(y_te, yp_lr, average="macro", zero_division=0)
    print(f"  [C] LR   acc={acc_lr:.4f}  F1={f1_lr:.4f}")

    print(f"\n  [D] Ensemble Uncertainty:")
    ens = EnsembleUncertainty().fit(X_sm_m, y_sm_m)
    ens_p, ens_ep, _ = ens.predict_with_uncertainty(X_te_m)
    yp_ens  = ens_p.argmax(1)
    acc_ens = accuracy_score(y_te, yp_ens)
    f1_ens  = f1_score(y_te, yp_ens, average="macro", zero_division=0)
    print(f"  [D] Ensemble acc={acc_ens:.4f}  F1={f1_ens:.4f}")

    print(f"\n  [E] Phantom/AR sub-classifier:")
    sub_clf = PhantomARSubClassifier().fit(X_sm_sb, y_sm_sb)

    idx_tr2, idx_val_i = train_test_split(
        np.arange(len(idx_tr)), test_size=0.15, stratify=y[idx_tr], random_state=RANDOM_SEED)
    X_rf_val  = X_rf[idx_tr][idx_val_i]; y_rf_val = y[idx_tr][idx_val_i]
    rf_val_proba = rf.predict_proba(X_rf_val)
    ts_cal = TemperatureScaler().fit(np.log(rf_val_proba.clip(1e-9, 1)), y_rf_val)
    cal_p  = ts_cal.calibrate(np.log(rf.predict_proba(X_te_rf).clip(1e-9, 1)))
    ece    = ts_cal.expected_calibration_error(cal_p, y_te)
    print(f"  ECE (RF, test)={ece:.4f}")

    print(f"\n  [FIX-4] Training stacking meta-learner ...")
    gbp_for_stack = GaussianBayesPosterior().fit(X_sm_m, y_sm_m)
    rf_p_te  = rf.predict_proba(X_te_rf)
    gbt_p_te = gbt.predict_proba(X_te_gbt)
    gbp_p_te = gbp_for_stack.predict_proba(X_te_m)

    stacker = StackingMetaLearner()
    stacker.fit(rf_p_te, gbt_p_te, gbp_p_te, y_te)

    stack_pred = np.array([
        stacker.predict_proba(rf_p_te[i], gbt_p_te[i], gbp_p_te[i])
        for i in range(len(y_te))
    ])
    acc_stack = accuracy_score(y_te, stack_pred.argmax(1))
    f1_stack  = f1_score(y_te, stack_pred.argmax(1), average="macro", zero_division=0)
    print(f"  [FIX-4] Stacking test acc={acc_stack:.4f}  F1={f1_stack:.4f}")

    return {
        "rf": rf, "gbt": gbt, "lr": lr_clf, "ens": ens,
        "sub_clf": sub_clf, "ts": ts_cal, "cnn": cnn,
        "stacker": stacker,
        "gbp_for_stack": gbp_for_stack,
        "X_te_m": X_te_m, "y_te": y_te,
        "X_te_rf": X_te_rf, "y_te_rf": y_te,
        "X_te_gbt": X_te_gbt, "y_te_gbt": y_te,
        "X_te_sub": X_te_sub, "y_te_sub": y_te,
        "X_te_raw": X_te_raw,
        "X_sm_m": X_sm_m, "y_sm": y_sm_m,
        "X_sm_sub": X_sm_sb, "y_sm_sub": y_sm_sb,
        "acc_rf": acc_rf,   "f1_rf": f1_rf,
        "acc_gbt": acc_gbt, "f1_gbt": f1_gbt,
        "acc_lr": acc_lr,   "f1_lr": f1_lr,
        "acc_ens": acc_ens, "f1_ens": f1_ens,
        "acc_stack": acc_stack, "f1_stack": f1_stack,
        "mean_ens_ep": float(ens_ep.mean()), "ece": ece,
        "rf_proba_te": rf_p_te, "y_te_rf": y_te,
    }


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 11 · SOFT FUSION ENGINE
# ─────────────────────────────────────────────────────────────────────────────
class SoftFusionEngine:
    def __init__(self, router, rf, gbt, gbp, ens, cnn, osd, ts_det, laplace, ts_cal,
                 sub_clf, classes, open_thr=0.35, friendly_thr=0.55):
        self.router=router; self.rf=rf; self.gbt=gbt; self.gbp=gbp
        self.ens=ens; self.cnn=cnn; self.osd=osd; self.ts_det=ts_det
        self.laplace=laplace; self.ts_cal=ts_cal; self.sub_clf=sub_clf
        self.classes=classes; self.n=len(classes)
        self.open_set_threshold=open_thr; self.friendly_threshold=friendly_thr
        self.hold_dead_band=HOLD_DEAD_BAND
        self.stacker: Optional[StackingMetaLearner] = None
        self.calibration_info = {
            "method": "default (uncalibrated)",
            "open_set_threshold": round(open_thr, 4),
            "friendly_threshold": round(friendly_thr, 4),
            "decision_threshold": round((open_thr + friendly_thr) / 2.0, 4),
            "hold_dead_band": round(HOLD_DEAD_BAND, 4),
        }

    def calibrate_thresholds_roc(self, X_raw_val, y_val, classes_present):
        print(f"\n  [v32-FIX2] Threshold calibration  ({len(X_raw_val)} val samples) ...")
        scores = []; drone_scores = []; bg_scores = []
        for i in range(len(X_raw_val)):
            sc = self.score(X_raw_val[i])
            ss = sc["soft_score"]
            scores.append(ss)
            if y_val[i] != 0: drone_scores.append(ss)
            else:              bg_scores.append(ss)

        arr       = np.array(scores)
        drone_arr = np.array(drone_scores) if drone_scores else arr
        bg_arr    = np.array(bg_scores)    if bg_scores    else arr

        open_thr     = float(np.percentile(drone_arr, DRONE_OPEN_SET_PERCENTILE))
        open_thr     = max(open_thr, float(np.percentile(bg_arr, OPEN_SET_FLOOR_PERCENTILE)))
        open_thr     = min(open_thr, OPEN_SET_THRESHOLD_CAP)
        friendly_thr = float(np.percentile(drone_arr, FRIENDLY_PERCENTILE))
        friendly_thr = max(friendly_thr, open_thr + FRIENDLY_MIN_GAP)
        friendly_thr = min(friendly_thr, float(np.percentile(arr, 95)))
        gap  = friendly_thr - open_thr
        dead = max(HOLD_DEAD_BAND, gap * 0.15)

        hold_frac     = float(((arr > open_thr + dead) & (arr < friendly_thr - dead)).mean())
        open_frac_val = float((arr < open_thr).mean())

        self.open_set_threshold = open_thr
        self.friendly_threshold = friendly_thr
        self.hold_dead_band     = dead

        self.calibration_info = {
            "method": "v32-FIX2",
            "open_set_threshold":    round(open_thr, 4),
            "friendly_threshold":    round(friendly_thr, 4),
            "decision_threshold":    round(self.decision_threshold(), 4),
            "hold_dead_band":        round(dead, 4),
            "hold_fraction_val":     round(hold_frac, 4),
            "open_set_fraction_val": round(open_frac_val, 4),
        }
        print(f"    open_thr={open_thr:.4f}  friendly_thr={friendly_thr:.4f}  "
              f"dead={dead:.4f}  hold≈{hold_frac:.1%}  open≈{open_frac_val:.1%}")
        return open_thr, friendly_thr

    def decision_threshold(self):
        return (self.open_set_threshold + self.friendly_threshold) / 2.0

    def _apply_cost_bias(self, combined, max_clf_prob):
        if not COST_BIAS_ACTIVE: return combined
        if max_clf_prob >= COST_BIAS_UNCERTAINTY_THR: return combined
        bg_idx = next((i for i, c in enumerate(self.classes) if c == BG_NAME), None)
        if bg_idx is None: return combined
        if int(np.argmax(combined)) != bg_idx: return combined
        combined = combined.copy()
        combined[bg_idx] = max(combined[bg_idx] - COST_BIAS_BG_PENALTY, 1e-6)
        combined /= combined.sum()
        return combined

    def score(self, fv_raw):
        if fv_raw.ndim == 1: fv_raw = fv_raw.reshape(1, -1)
        fv_raw = np.nan_to_num(fv_raw.astype(np.float32), nan=0., posinf=0., neginf=0.)
        eps = 1e-12

        fv_rf    = fv_raw.ravel()[self.router.rf_idx]
        rf_p     = self.rf.predict_proba(fv_rf.reshape(1, -1))[0]
        max_rf_p = float(rf_p.max())

        if max_rf_p > RF_FAST_PATH_THRESHOLD:
            win_idx  = int(rf_p.argmax())
            fp_soft  = float(max_rf_p * 0.82)
            return {
                "winner": self.classes[win_idx], "winner_idx": win_idx,
                "combined_probs": rf_p.round(4).tolist(),
                "clf_conf": round(max_rf_p, 4), "cnn_conf": 0.0,
                "evm_score": 1.0, "normality": 1.0, "anomaly_raw": 0.0,
                "agreement_score": 1.0, "ens_epistemic": 0.0, "ens_aleatoric": 0.0,
                "predictive_entropy": 0.0, "sub_boost": 0.0,
                "soft_score": round(fp_soft, 4), "margin": 1.0,
                "threat_score": 0.0, "max_clf_prob": round(max_rf_p, 4),
                "decision_threshold": round(self.decision_threshold(), 4),
                "is_novel": False,
                "open_set_threshold": round(self.open_set_threshold, 4),
                "friendly_threshold": round(self.friendly_threshold, 4),
                "bayesian": {},
            }

        cached = _ROUTE_CACHE.get(fv_raw.ravel())
        if cached is not None:
            routed = cached
        else:
            routed = self.router.route(fv_raw)
            _ROUTE_CACHE.put(fv_raw.ravel(), routed)

        gbt_p = self.gbt.predict_proba(routed["gbt"])[0].astype(np.float64) + eps
        gbp_p = self.gbp.predict_proba(routed["master"])[0].astype(np.float64) + eps
        cnn_p = self.cnn.predict_proba(fv_raw)[0].astype(np.float64) + eps
        cnn_p /= cnn_p.sum()

        if (self.stacker is not None and self.stacker.fitted):
            combined = self.stacker.predict_proba(
                rf_p.astype(np.float64) + eps, gbt_p, gbp_p,
            ).astype(np.float64)
            combined = np.clip(combined, eps, None)
            combined /= combined.sum()
        else:
            combined = (rf_p.astype(np.float64) * gbt_p * gbp_p) ** (1 / 3)
            combined /= combined.sum()

        combined  = self._apply_cost_bias(combined, float(combined.max()))
        win_idx   = int(combined.argmax())
        sorted_c  = np.sort(combined)[::-1]
        margin    = float(sorted_c[0] - sorted_c[1]) if self.n > 1 else 1.

        stacked   = np.stack([rf_p / rf_p.sum(), gbt_p / gbt_p.sum(),
                               gbp_p / gbp_p.sum(), cnn_p], 0)
        agreement_score = float(np.clip(1. - stacked.std(0).mean() * self.n, 0., 1.))

        cal_p     = self.ts_cal.calibrate(np.log(rf_p.clip(1e-9, 1)).reshape(1, -1))[0]
        clf_conf  = float(cal_p.max() * (0.5 + 0.5 * margin))
        evm_score = float(self.osd.inclusion_score(routed["master"])[0])
        anomaly_raw = float(self.ts_det.compute(routed["master"])[0])
        normality   = float(1. - np.clip(anomaly_raw, 0., 1.))

        ens_probs, ens_ep, ens_al = self.ens.predict_with_uncertainty(routed["master"])
        ens_vacuity = float(np.clip(ens_ep[0] * 5., 0., 1.))
        norm_H      = float(-np.dot(combined, np.log(combined + eps)) / (np.log(self.n) + eps))

        sub_boost = 0.0
        if self.sub_clf.fitted and self.n > 2:
            ar_idx = next((i for i, c in enumerate(self.classes) if "AR" in c), None)
            ph_idx = next((i for i, c in enumerate(self.classes) if "Phantom" in c), None)
            if ar_idx is not None and ph_idx is not None:
                if float(combined[ar_idx]) + float(combined[ph_idx]) > 0.40:
                    p_ph   = self.sub_clf.p_phantom(routed["sub"])
                    delta  = (p_ph - 0.5) * 0.30
                    combined[ar_idx] = float(np.clip(combined[ar_idx] - delta, eps, 1.))
                    combined[ph_idx] = float(np.clip(combined[ph_idx] + delta, eps, 1.))
                    combined /= combined.sum()
                    win_idx   = int(combined.argmax())
                    sub_boost = abs(delta)

        raw_soft   = (FUSION_W_CLF * clf_conf + FUSION_W_CNN * float(cnn_p.max()) +
                      FUSION_W_EVM * evm_score + FUSION_W_NORMALITY * normality +
                      FUSION_W_AGREEMENT * agreement_score)
        soft_score = float(raw_soft * float(np.clip(1. - ens_vacuity * 0.3, 0.70, 1.0)))

        return {
            "winner": self.classes[win_idx], "winner_idx": win_idx,
            "combined_probs": combined.round(4).tolist(),
            "clf_conf": round(clf_conf, 4), "cnn_conf": round(float(cnn_p.max()), 4),
            "evm_score": round(evm_score, 4), "normality": round(normality, 4),
            "anomaly_raw": round(anomaly_raw, 4), "agreement_score": round(agreement_score, 4),
            "ens_epistemic": round(ens_vacuity, 4), "ens_aleatoric": round(float(ens_al[0]), 4),
            "predictive_entropy": round(norm_H, 4), "soft_score": round(soft_score, 4),
            "margin": round(margin, 4), "threat_score": round(anomaly_raw, 4),
            "sub_boost": round(sub_boost, 4),
            "max_clf_prob": round(float(combined.max()), 4),
            "decision_threshold": round(self.decision_threshold(), 4),
            "is_novel": bool(anomaly_raw > self.open_set_threshold),
            "open_set_threshold": round(self.open_set_threshold, 4),
            "friendly_threshold": round(self.friendly_threshold, 4),
            "bayesian": {},
        }


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 11b · FINGERPRINT DB + TEMPORAL TRACKER
# ─────────────────────────────────────────────────────────────────────────────
def emitter_hash(fv):
    qfp = np.round(fv / 0.05).astype(np.int32)
    stable_features = qfp[_HASH_IDX[0]]
    return hashlib.blake2b(stable_features.tobytes(), digest_size=8).hexdigest()

def cosine_sim(a, b):
    a=a.ravel().astype(np.float64); b=b.ravel().astype(np.float64)
    return float(np.dot(a,b)/((np.dot(a,a)*np.dot(b,b))**0.5+1e-12))


@dataclass
class EmitterRecord:
    emitter_id: str
    feature_history: deque = field(default_factory=lambda: deque(maxlen=50))
    first_seen: float = field(default_factory=time.time)
    last_seen:  float = field(default_factory=time.time)
    seen_count: int = 0
    threat_scores: List[float] = field(default_factory=list)
    soft_scores:   List[float] = field(default_factory=list)
    label_history: List[str]   = field(default_factory=list)
    trust_score: float = 0.
    promoted:    bool  = False
    auto_class:  Optional[str] = None
    auto_conf:   float = 0.
    promotion_time: Optional[float] = None
    time_to_trust_s: Optional[float] = None

    def update(self, fv, ts, ss, label=None):
        self.feature_history.append(fv.copy()); self.last_seen=time.time()
        self.seen_count+=1; self.threat_scores.append(float(ts)); self.soft_scores.append(float(ss))
        if label is not None: self.label_history.append(label)

    @property
    def mean_features(self): return np.mean(np.stack(list(self.feature_history)),0)

    @property
    def feature_variance(self):
        if len(self.feature_history)<2: return 1.
        stack=np.stack(list(self.feature_history)); stds=stack.std(0)+1e-9
        return float(np.mean((stack/stds).var(0)))

    @property
    def mean_threat(self): return float(np.mean(self.threat_scores)) if self.threat_scores else 1.

    def majority_vote_label(self):
        if len(self.label_history)<TEMPORAL_SMOOTHING_MIN: return None
        recent=list(self.label_history)[-TEMPORAL_WINDOW:]
        if not recent: return None
        ctr=Counter(recent); winner,count=ctr.most_common(1)[0]
        if count/len(recent)>=0.40: return winner
        return None

    def compute_trust(self):
        obs_t  = float(1/(1+np.exp(-(self.seen_count-TRUST_MIN_OBSERVATIONS)/3)))
        stab_t = float(max(0., 1. - self.feature_variance / (TRUST_MAX_VARIANCE + 1e-9)))
        safe_t = float(max(0., 1. - self.mean_threat))
        vals   = [obs_t, stab_t, safe_t]
        self.trust_score = float(np.clip(len(vals)/sum(1/(v+1e-9) for v in vals), 0., 1.))
        return self.trust_score

    def is_trustworthy(self):
        return (self.seen_count >= TRUST_MIN_OBSERVATIONS and
                self.feature_variance <= TRUST_MAX_VARIANCE and
                self.mean_threat < HIGH_THREAT_THRESHOLD)

    def is_promotion_eligible(self):
        return (self.seen_count >= PROMO_MIN_OBS and
                self.trust_score >= PROMO_TRUST_THR and
                self.mean_threat < PROMO_MAX_THREAT and
                not self.promoted)


class TemporalTracker:
    def __init__(self):
        self.registry={}; self.total_obs=0
        self.promo_times: List[float] = []

    def observe(self, fv, ts, ss=0.5, label=None):
        eid=emitter_hash(fv)
        if eid not in self.registry:
            self.registry[eid]=EmitterRecord(emitter_id=eid, first_seen=time.time())
        rec=self.registry[eid]; rec.update(fv,ts,ss,label); rec.compute_trust()
        self.total_obs+=1; return rec

    def get_record(self, fv):
        eid=emitter_hash(fv)
        return self.registry.get(eid, None)

    def record_promotion(self, rec: EmitterRecord):
        now = time.time()
        rec.promoted = True; rec.promotion_time = now
        rec.time_to_trust_s = now - rec.first_seen
        self.promo_times.append(rec.time_to_trust_s)

    def mean_time_to_trust(self) -> float:
        return float(np.mean(self.promo_times)) if self.promo_times else float("nan")

    def reset(self):
        self.registry={}; self.total_obs=0; self.promo_times=[]

    def summary(self):
        n   = len(self.registry)
        nt  = sum(1 for r in self.registry.values() if r.is_trustworthy())
        nth = sum(1 for r in self.registry.values() if r.mean_threat>=HIGH_THREAT_THRESHOLD)
        np_ = sum(1 for r in self.registry.values() if r.promoted)
        t2t = self.mean_time_to_trust()
        t2t_str = f"{t2t:.1f}s" if not np.isnan(t2t) else "N/A"
        return (f"Tracker: {n} emitters | trustworthy={nt} | "
                f"promoted={np_} | threat={nth} | TTT={t2t_str}")


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 11b-NEW · [NEW-3] TRACKER STABILITY UNIT TEST
# ─────────────────────────────────────────────────────────────────────────────
def test_tracker_stability(n_obs: int = 10, cls: int = 1,
                            noise_scale: float = 0.1, verbose: bool = True) -> bool:
    """
    [NEW-3] Verifies the TemporalTracker promotes a known drone after
    observing it n_obs times.

    Prints a per-observation trust log so you can see exactly when the
    system transitions from "observing" to "trustworthy":

      Obs 1: Seen=1 | Trust=0.10 | Trustworthy=False   ← still learning
      Obs 4: Seen=4 | Trust=0.72 | Trustworthy=True    ← PROMOTED

    If Trustworthy stays False after n_obs:
      • Variance is too high → increase TRUST_MAX_VARIANCE (currently 0.90)
      • Or n_obs is below TRUST_MIN_OBSERVATIONS (currently 4)
    If Trustworthy is True from obs=1:
      • emitter_hash() is too loose → different signals map to the same ID
    """
    if _HASH_IDX[0] is None:
        # Bootstrap with a minimal index so emitter_hash works standalone
        _HASH_IDX[0] = np.arange(HASH_TOP_FEATURES, dtype=np.int64)

    tracker = TemporalTracker()
    rng     = np.random.default_rng(RANDOM_SEED + 42)

    # Generate a single "template" burst for class `cls`
    fake_drone = _generate_rf_burst(cls, rng, noise_scale=noise_scale)
    cls_name   = CLASS_NAMES.get(cls, str(cls))

    if verbose:
        print(f"\n{'─'*60}")
        print(f"  [NEW-3] Tracker Stability Test  "
              f"(cls={cls_name}, n_obs={n_obs}, noise={noise_scale})")
        print(f"{'─'*60}")

    for i in range(n_obs):
        # Add tiny jitter so feature_variance is non-zero (realistic)
        jittered = fake_drone + rng.normal(0, 1e-3, fake_drone.shape).astype(np.float32)
        rec = tracker.observe(jittered, ts=0.05, ss=0.8, label=cls_name)
        rec.compute_trust()

        # ── [NEW-3] Per-observation audit log ────────────────────────────────
        line = (f"  Obs {i+1:>2}: "
                f"Seen={rec.seen_count:>2} | "
                f"Variance={rec.feature_variance:.3f} | "
                f"Trust={rec.trust_score:.2f} | "
                f"Trustworthy={rec.is_trustworthy()}")
        if verbose:
            print(line)
        audit("TRACKER_STABILITY_OBS",
              obs=i+1, seen=rec.seen_count,
              variance=round(rec.feature_variance, 4),
              trust=round(rec.trust_score, 4),
              trustworthy=rec.is_trustworthy())

    final_rec = tracker.get_record(fake_drone)
    success   = (final_rec is not None and final_rec.is_trustworthy())

    if verbose:
        if success:
            print(f"\n  ✅ SUCCESS: Tracker trusts emitter after {n_obs} observations.")
            print(f"     Final trust_score={final_rec.trust_score:.4f}  "
                  f"variance={final_rec.feature_variance:.4f}")
        else:
            print(f"\n  ❌ FAIL: Tracker does NOT trust the emitter after {n_obs} observations.")
            if final_rec:
                print(f"     trust_score={final_rec.trust_score:.4f}  "
                      f"variance={final_rec.feature_variance:.4f}  "
                      f"seen={final_rec.seen_count}")
                if final_rec.feature_variance > TRUST_MAX_VARIANCE:
                    print(f"     → variance ({final_rec.feature_variance:.3f}) "
                          f"> TRUST_MAX_VARIANCE ({TRUST_MAX_VARIANCE}) — increase it.")
                if final_rec.seen_count < TRUST_MIN_OBSERVATIONS:
                    print(f"     → seen_count ({final_rec.seen_count}) "
                          f"< TRUST_MIN_OBSERVATIONS ({TRUST_MIN_OBSERVATIONS}) — "
                          f"increase n_obs.")
        print(f"{'─'*60}\n")

    return success


class FingerprintDatabase:
    def __init__(self, path):
        self.path=path; self.trusted={}; self.suspicious={}
        self._load()
        self.total_queries = 0
        self.memory_hits   = 0

    def _load(self):
        if Path(self.path).exists():
            try:
                d=json.load(open(self.path))
                self.trusted=d.get("trusted",{}); self.suspicious=d.get("suspicious",{})
                print(f"  DB: {len(self.trusted)} trusted, {len(self.suspicious)} suspicious")
            except: print("  DB corrupted → fresh")
        else: print("  DB: starting fresh")

    def save(self):
        json.dump({"trusted":self.trusted,"suspicious":self.suspicious},
                  open(self.path,"w"), indent=2)

    def reset(self):
        self.trusted={}; self.suspicious={}
        self.total_queries=0; self.memory_hits=0

    def hit_rate(self) -> float:
        if self.total_queries == 0: return 0.
        return self.memory_hits / self.total_queries

    def lookup(self, eid: str) -> Optional[dict]:
        self.total_queries += 1
        rec = self.trusted.get(eid, None)
        if rec is not None: self.memory_hits += 1
        return rec

    def match(self, fv):
        best_sim,best_id,best_store=-1.,None,""
        for sname,db in (("trusted",self.trusted),("suspicious",self.suspicious)):
            for eid,rec in db.items():
                sim=cosine_sim(fv,np.array(rec["fingerprint"]))
                if sim>best_sim: best_sim,best_id,best_store=sim,eid,sname
        return best_id,float(best_sim),best_store

    def add_trusted(self, eid, fv, seen, pred_class, conf):
        is_new=eid not in self.trusted
        if conf>=AUTO_CLASSIFY_CONF and pred_class!=BG_NAME:
            label=f"AUTO_{pred_class.upper().replace(' ','_')}"
        elif is_new: label=f"SAFE_UNKNOWN_{len(self.trusted)+1:03d}"
        else: label=self.trusted[eid]["label"]
        self.trusted[eid]={"fingerprint":fv.tolist(),"label":label,
            "predicted_class":pred_class,"confidence":round(conf,4),
            "seen_count":seen,"last_updated":time.time(),
            "first_seen":self.trusted[eid]["first_seen"] if not is_new else time.time()}
        self.save()

    def add_suspicious(self, eid, fv, seen=0):
        if eid not in self.suspicious:
            self.suspicious[eid]={"fingerprint":fv.tolist(),
                "label":f"THREAT_{len(self.suspicious)+1:03d}","seen_count":seen,"added_at":time.time()}
        else: self.suspicious[eid]["seen_count"]=seen
        self.save()

    def summary(self):
        return (f"DB: {len(self.trusted)} trusted | {len(self.suspicious)} suspicious "
                f"| hit_rate={self.hit_rate():.1%} ({self.memory_hits}/{self.total_queries})")


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 11c · PRE-SEED DB
# ─────────────────────────────────────────────────────────────────────────────
def preseed_fingerprint_db(fp_db, tracker, X_raw_tr, y_tr, classify_signal,
                            classes_present, n_per_class=PRESEED_N_PER_CLASS):
    print(f"\n  [FIX-3/6] Pre-seeding fingerprint DB  ({n_per_class}/class) ...")
    seeded = 0
    rng    = np.random.default_rng(RANDOM_SEED + 7)
    for cls_idx, cls_name in enumerate(classes_present):
        cls_mask = (y_tr == cls_idx)
        cls_rows = X_raw_tr[cls_mask]
        if len(cls_rows) == 0: continue
        sample_idx = rng.choice(len(cls_rows),
                                 size=min(n_per_class, len(cls_rows)),
                                 replace=False)
        for si in sample_idx:
            _ = classify_signal(cls_rows[si])
            seeded += 1
    print(f"    Seeded {seeded} signals ({len(fp_db.trusted)} trusted entries in DB)")
    return fp_db


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 12 · FAIL-SAFE GUARD
# ─────────────────────────────────────────────────────────────────────────────
class FailSafeGuard:
    def check(self, rec, label, soft_score, open_thr, hold_dead=HOLD_DEAD_BAND,
              max_clf_prob=0., threat_score=0., decision_threshold=0.):
        if label=="FRIENDLY_DRONE" and max_clf_prob>0.90: return label
        bypass_ok=(max_clf_prob>CONFIDENCE_BYPASS_THRESHOLD and
                   threat_score<open_thr*CONFIDENCE_BYPASS_THREAT_RATIO)
        if bypass_ok: return label
        if abs(soft_score-decision_threshold)<hold_dead: return "HOLD"
        return label


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 12b · HYSTERESIS LAYER
# ─────────────────────────────────────────────────────────────────────────────
class HysteresisFilter:
    def __init__(self, fn, window=HYSTERESIS_WINDOW, majority=HYSTERESIS_MAJORITY):
        self.fn=fn; self.window=window; self.majority=majority
        self._buffers: Dict[str, deque] = defaultdict(lambda: deque(maxlen=window))
        self._ui_labels: Dict[str, str] = {}

    def reset(self):
        self._buffers.clear(); self._ui_labels.clear()

    def classify(self, fv_raw, return_bayes=True):
        result  = self.fn(fv_raw, return_bayes=return_bayes)
        eid     = result.get("emitter_id", "unknown")
        raw_lbl = result.get("label", "HOLD")
        source  = result.get("source", "CLASSIFIER")

        if source == "MEMORY_MATCH":
            self._ui_labels[eid] = raw_lbl
            self._buffers[eid].append(raw_lbl)
            result["ui_label"] = raw_lbl
            result["label"]    = raw_lbl
            return result

        buf = self._buffers[eid]; buf.append(raw_lbl)
        if len(buf) == 1:
            self._ui_labels[eid] = raw_lbl
            result["ui_label"]  = raw_lbl; result["raw_label"] = raw_lbl
            result["label_votes"] = {raw_lbl: 1}; result["label"] = raw_lbl
            return result

        votes = Counter(buf); top_lbl, top_cnt = votes.most_common(1)[0]
        current_ui = self._ui_labels.get(eid, raw_lbl)
        required = self.majority if len(buf) >= self.window else max(2, len(buf)//2+1)
        if top_cnt >= required and top_lbl != current_ui:
            self._ui_labels[eid] = top_lbl
        elif eid not in self._ui_labels:
            self._ui_labels[eid] = raw_lbl

        result["ui_label"]    = self._ui_labels[eid]
        result["raw_label"]   = raw_lbl
        result["label_votes"] = dict(votes)
        result["label"]       = self._ui_labels[eid]
        return result


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 12c · CLASSIFY FUNCTION  [NEW-1 + NEW-3 integrated]
# ─────────────────────────────────────────────────────────────────────────────
def make_classify_fn(fusion, fp_db, tracker, classes_present, threat_scorer, failsafe,
                      action_ctrl: Optional["ActionController"] = None):
    """
    Returns classify_signal(fv_raw, return_bayes=True) → dict.

    [NEW-1] If `action_ctrl` is supplied and the final label is a threat,
            action_ctrl.trigger_defense() is called automatically.

    [NEW-3] Every call logs the emitter's Trust state so you can monitor
            when the system promotes an emitter from "observing" to "trusted".
            Look for DEBUG lines in antidrone_audit_v32.jsonl:
              {"event": "TRUST_STATE", "seen": N, "trust": X, "trustworthy": bool}
    """
    def classify_signal(fv_raw, return_bayes=True):
        t0 = time.perf_counter()
        fv = np.nan_to_num(fv_raw.astype(np.float32).ravel(), nan=0., posinf=0., neginf=0.)
        if len(fv) < N_FEATURES:
            pad = np.zeros(N_FEATURES, dtype=np.float32); pad[:len(fv)] = fv; fv = pad
        fv = fv[:N_FEATURES]
        eid = emitter_hash(fv)

        db_rec = fp_db.lookup(eid)
        if db_rec:
            label = db_rec.get("label", "TRUSTED_NEW_DRONE")
            tracker.observe(fv, ts=0.0, ss=0.9, label=label)
            return {"label": label, "bayesian": {}, "emitter_id": eid,
                    "soft_score": 0.9, "source": "MEMORY_MATCH", "bypass_used": False,
                    "latency_ms": round((time.perf_counter()-t0)*1000, 3)}

        rec = tracker.observe(fv, ts=0.1, ss=0.5, label=None)
        sc  = fusion.score(fv)
        ss  = sc["soft_score"]; ts_val = sc["threat_score"]; mcp = sc.get("max_clf_prob", 0.)
        winner = sc["winner"]
        rec.threat_scores[-1] = ts_val; rec.soft_scores[-1] = ss; rec.compute_trust()

        # ── [NEW-3] Real-time trust audit log ────────────────────────────────
        trust_log = (f"  TRUST  ID={eid[:6]} | Label={winner} | "
                     f"Seen={rec.seen_count} | "
                     f"Var={rec.feature_variance:.3f} | "
                     f"Trust={rec.trust_score:.2f} | "
                     f"Trustworthy={rec.is_trustworthy()}")
        if not PRODUCTION_MODE:
            print(trust_log)
        audit("TRUST_STATE",
              emitter_id=eid[:8], label=winner,
              seen=rec.seen_count,
              variance=round(rec.feature_variance, 4),
              trust=round(rec.trust_score, 4),
              trustworthy=rec.is_trustworthy())

        base = {"bayesian": sc if return_bayes else {}, "emitter_id": eid,
                "soft_score": round(ss,4), "bypass_used": False, "source": "CLASSIFIER"}

        def _ret(label, source=None, bypass=False):
            r = dict(base); r["label"]=label; r["bypass_used"]=bypass
            r["source"]=source or base["source"]
            r["latency_ms"]=round((time.perf_counter()-t0)*1000, 3)
            # ── [NEW-1] Action trigger ────────────────────────────────────────
            if action_ctrl is not None and label in THREAT_LABELS:
                action_ctrl.trigger_defense(
                    threat_label=label, emitter_id=eid,
                    soft_score=r["soft_score"])
            return r

        amp_mean      = float(fv[FEAT_IDX["amp_mean"]])
        I_power       = float(fv[FEAT_IDX["I_power"]])
        Q_power       = float(fv[FEAT_IDX["Q_power"]])
        signal_pwr_db = float(fv[FEAT_IDX["signal_power_db"]])
        spectral_entr = float(fv[FEAT_IDX["spectral_entropy"]])
        iq_ratio      = float(fv[FEAT_IDX["iq_power_ratio"]])

        is_physically_impossible = (
            amp_mean<=0. or I_power<=0. or Q_power<=0. or
            signal_pwr_db<-60. or spectral_entr>9. or
            iq_ratio<=0. or iq_ratio>50.)

        fv_norm = float(np.linalg.norm(fv)); fv_std = float(np.std(fv))
        rf_max  = float(np.abs(fv[:N_RF]).max())
        is_structurally_weak = (fv_norm<1. or rf_max<0.10 or (fv_std<0.05 and fv_norm<3.))

        if (mcp<0.20 or is_structurally_weak or is_physically_impossible):
            rec.label_history.append("OPEN_SET_UNKNOWN")
            return _ret("OPEN_SET_UNKNOWN", source="NOISE_REJECTION")

        if ss < fusion.open_set_threshold:
            rec.label_history.append("OPEN_SET_UNKNOWN")
            return _ret("OPEN_SET_UNKNOWN", source="SVDD_GATE")

        if rec.is_promotion_eligible() and mcp >= PROMO_CONF_THR:
            fp_db.add_trusted(eid, rec.mean_features, rec.seen_count, winner, mcp)
            tracker.record_promotion(rec)
            rec.label_history.append(f"AUTO_{winner.upper()}")
            return _ret(f"AUTO_{winner.upper()}", source="PROMOTED")

        if winner != BG_NAME and (mcp > 0.15 or rec.seen_count > 2):
            final_label = "FRIENDLY_DRONE"
        elif ts_val > HIGH_THREAT_THRESHOLD:
            final_label = "POTENTIAL_THREAT"
        else:
            final_label = "BACKGROUND"

        rec.label_history.append(final_label)
        return _ret(final_label)

    return classify_signal


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 13b · THREE PROFESSIONAL STRESS-TESTS
# ─────────────────────────────────────────────────────────────────────────────
def run_stress_tests(classify_signal, fp_db, tracker, fusion, router,
                     classes_present, rng_seed=RANDOM_SEED):
    print(f"\n{'═'*65}\n  [M4] PROFESSIONAL STRESS-TESTS\n{'═'*65}")
    rng = np.random.default_rng(rng_seed + 99); results = {}

    print("\n  [A] Ghost Hunt")
    fv_phantom = _generate_rf_burst(2, rng, noise_scale=0.5)
    eid_phantom = emitter_hash(fv_phantom)
    fp_db.trusted[eid_phantom] = {"fingerprint": fv_phantom.tolist(),
        "label": "AUTO_PHANTOM_DRONE", "predicted_class": "Phantom Drone",
        "confidence": 0.99, "seen_count": 10,
        "first_seen": time.time(), "last_updated": time.time()}
    transitions=0; prev_lbl=None; labels_seen=[]
    for _ in range(GHOST_HUNT_BURSTS):
        noisy = fv_phantom + rng.normal(0, 1e-4, fv_phantom.shape).astype(np.float32)
        dec   = classify_signal(noisy); lbl = dec["label"]; labels_seen.append(lbl)
        if prev_lbl is not None and lbl != prev_lbl: transitions += 1
        prev_lbl = lbl
    ghost_pass = (transitions == 0)
    print(f"    Transitions={transitions}  Labels={dict(Counter(labels_seen))}")
    print(f"    {'✅ PASS' if ghost_pass else '❌ FAIL'}")
    results["ghost_hunt"] = {"bursts": GHOST_HUNT_BURSTS, "transitions": transitions,
        "label_distribution": dict(Counter(labels_seen)), "pass": ghost_pass}

    print(f"\n  [B] Adversarial")
    adv_labels = []
    for _ in range(ADVERSARIAL_SAMPLES):
        noise_fv = np.random.default_rng().uniform(-1,1,N_FEATURES).astype(np.float32)
        adv_labels.append(classify_signal(noise_fv)["label"])
    safe_labels   = {"OPEN_SET_UNKNOWN", "BACKGROUND", "HOLD"}
    adv_safe_rate = sum(1 for l in adv_labels if l in safe_labels) / ADVERSARIAL_SAMPLES
    adv_pass = adv_safe_rate >= 0.90
    print(f"    Safe_rate={adv_safe_rate:.1%}  Labels={dict(Counter(adv_labels).most_common(3))}")
    print(f"    {'✅ PASS' if adv_pass else '❌ FAIL'}")
    results["adversarial"] = {"samples": ADVERSARIAL_SAMPLES, "safe_rate": round(adv_safe_rate,4),
        "label_distribution": dict(Counter(adv_labels)), "pass": adv_pass}

    print(f"\n  [C] Recovery Time")
    fv_ar = _generate_rf_burst(1, rng, noise_scale=1.0)
    fv_ar[FEAT_IDX["ifreq_std"]] += 999.0
    eid_ar = emitter_hash(fv_ar); fp_db.trusted.pop(eid_ar, None)
    stable_label=None; stable_burst=None; burst_times_ms=[]; rec_labels=[]
    for burst_i in range(RECOVERY_BURST_COUNT):
        noisy = fv_ar + rng.normal(0, 0.01, fv_ar.shape).astype(np.float32)
        t_b = time.perf_counter(); dec = classify_signal(noisy)
        burst_times_ms.append((time.perf_counter()-t_b)*1000)
        lbl = dec["label"]; rec_labels.append(lbl)
        if (stable_label is None and burst_i>=3 and
                rec_labels[-1]==rec_labels[-2]==rec_labels[-3]):
            stable_label=rec_labels[-1]; stable_burst=burst_i+1
    ttt_s = (stable_burst * 50 / 1000) if stable_burst else float("nan")
    recovery_pass = (not np.isnan(ttt_s) and ttt_s <= GATE_TIME_TO_TRUST_S)
    print(f"    Stable at burst #{stable_burst}  TTT={ttt_s:.1f}s  "
          f"p95={np.percentile(burst_times_ms,95):.1f}ms")
    print(f"    {'✅ PASS' if recovery_pass else '❌ FAIL'}")
    results["recovery"] = {"burst_count": RECOVERY_BURST_COUNT,
        "stable_at_burst": stable_burst, "stable_label": stable_label,
        "simulated_ttt_s": round(ttt_s,2) if not np.isnan(ttt_s) else None,
        "p95_burst_ms": round(float(np.percentile(burst_times_ms,95)),2), "pass": recovery_pass}

    all_pass = all(r["pass"] for r in results.values())
    print(f"\n  {'🎉 All stress-tests passed' if all_pass else '⚠️  Some stress-tests failed'}")
    return results, all_pass


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 14 · DIAGNOSTICS
# ─────────────────────────────────────────────────────────────────────────────
def run_diagnostics(rf_clf, X_te_rf, y_te_rf, router, X_te_raw, test_df,
                    classes_present, diag_dir=DIAG_DIR):
    print(f"\n{'='*60}\nDIAGNOSTICS\n{'='*60}")
    os.makedirs(diag_dir, exist_ok=True)

    try:
        if LGB_OK and isinstance(rf_clf, LGBClassifier) and SHAP_OK:
            explainer = shap.TreeExplainer(rf_clf.booster)
            X_shap    = X_te_rf[:min(200, len(X_te_rf))]
            shap_vals = explainer.shap_values(X_shap)
            if isinstance(shap_vals, list):
                abs_shap = np.mean([np.abs(sv) for sv in shap_vals], axis=0)
            else:
                abs_shap = np.abs(shap_vals)
            mean_abs = abs_shap.mean(0); top_k = np.argsort(mean_abs)[::-1][:15]
            rf_feat_names = [ALL_FEATURE_NAMES[i] for i in router.rf_idx]
            fig, ax = plt.subplots(figsize=(8,5))
            top_names  = [rf_feat_names[i] if i<len(rf_feat_names) else f"f{i}" for i in top_k]
            ax.barh(range(len(top_k)), mean_abs[top_k][::-1], color="#378ADD")
            ax.set_yticks(range(len(top_k))); ax.set_yticklabels(top_names[::-1], fontsize=9)
            ax.set_xlabel("Mean |SHAP value|"); ax.set_title("Top-15 RF features (LGB SHAP)")
            plt.tight_layout()
            path = f"{diag_dir}/shap_lgb_rf.png"
            fig.savefig(path, dpi=120); plt.close(fig)
            print(f"  ✓ LGB SHAP saved → {path}")
    except Exception as e:
        print(f"  ⚠  SHAP failed: {e}")

    try:
        rf_proba_te = rf_clf.predict_proba(X_te_rf); n_bins=10
        fig, axes = plt.subplots(1, len(classes_present), figsize=(4*len(classes_present),4))
        if len(classes_present)==1: axes=[axes]
        for i, cls_name in enumerate(classes_present):
            ax=axes[i]; y_bin=(y_te_rf==i).astype(int); prob_cls=rf_proba_te[:,i]
            bin_edges=np.linspace(0,1,n_bins+1); bin_acc=[]; bin_conf=[]
            for lo,hi in zip(bin_edges[:-1],bin_edges[1:]):
                mask=(prob_cls>=lo)&(prob_cls<hi)
                if mask.sum()==0: continue
                bin_acc.append(y_bin[mask].mean()); bin_conf.append(prob_cls[mask].mean())
            ax.plot([0,1],[0,1],"--",color="#888",lw=1); ax.bar(bin_conf,bin_acc,width=0.08,alpha=0.6,color="#378ADD")
            ax.set_title(cls_name,fontsize=10); ax.set_xlabel("Confidence"); ax.set_ylabel("Accuracy")
            ax.set_xlim(0,1); ax.set_ylim(0,1)
        fig.suptitle("Calibration reliability (LGB-RF)", fontsize=11); plt.tight_layout()
        path = f"{diag_dir}/calibration_curves.png"
        fig.savefig(path, dpi=120); plt.close(fig)
        print(f"  ✓ Calibration curves → {path}")
    except Exception as e:
        print(f"  ⚠  Calibration curves failed: {e}")


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 15 · SELF-TEST SUITE  [NEW-1/2/3 assertions added]
# ─────────────────────────────────────────────────────────────────────────────
def run_self_tests(fusion, models, router, df, eval_results,
                   osd_detector=None, hysteresis_filter=None,
                   stress_results=None, action_ctrl=None):
    print(f"\n{'='*60}\nSELF-TEST SUITE  (v32-FIELD)\n{'='*60}")
    passed=0; failed=0

    def test(name, condition, msg=""):
        nonlocal passed, failed
        if condition: print(f"  ✅ PASS  {name}"); passed+=1
        else:         print(f"  ❌ FAIL  {name}  {msg}"); failed+=1

    rng = np.random.default_rng(0)

    test("T1:  N_FEATURES=83",       N_FEATURES == 83)
    test("T1b: HLBR in schema",      "high_low_band_ratio" in FEAT_IDX)

    fv_raw = _generate_rf_burst(1, rng)
    routed = router.route(fv_raw)
    test("T2a: RF shape",  routed["rf"].shape  == (1, RF_TOP_K_MI))
    test("T2b: GBT shape", routed["gbt"].shape == (1, GBT_TOP_K_VAR))

    try:
        p = models["rf"].predict_proba(routed["rf"])
        test("T3a: RF predict_proba", p.shape[1] == len(fusion.classes))
    except Exception as e:
        test("T3a: RF predict_proba", False, str(e))

    test("T4:  Temperature in range", TEMP_MIN <= models["ts"].T <= TEMP_MAX)

    test("T_FIX5: LGB_OK or sklearn fallback",     LGB_OK or True)
    test("T_FIX5: LGB_DEVICE defined",             _LGB_DEVICE in ("cpu","gpu"))
    if LGB_OK:
        rf_model = models.get("rf")
        test("T_FIX5: RF is LGBClassifier",        isinstance(rf_model, LGBClassifier))
        test("T_FIX5: RF booster fitted",           rf_model is not None and rf_model.booster is not None)
        gbt_model = models.get("gbt")
        test("T_FIX5: GBT is LGBClassifier",       isinstance(gbt_model, LGBClassifier))
        test("T_FIX5: CUDA flag defined",           isinstance(CUDA_OK, bool))
        if CUDA_OK:
            test("T_FIX5: DEVICE is cuda",         str(DEVICE) == "cuda")

    test("T_FIX6: TRUST_MAX_VARIANCE=0.90",        abs(TRUST_MAX_VARIANCE - 0.90) < 1e-9)
    test("T_FIX6: PRESEED_N_PER_CLASS=80",         PRESEED_N_PER_CLASS == 80)
    rec_test = EmitterRecord(emitter_id="test_var")
    rec_test.seen_count = 10
    for _ in range(10):
        fv_t = _generate_rf_burst(1, rng, noise_scale=2.5)
        rec_test.feature_history.append(fv_t)
    rec_test.threat_scores = [0.05]*10
    rec_test.compute_trust()
    test("T_FIX6: real-world variance passes is_trustworthy",
         rec_test.feature_variance <= TRUST_MAX_VARIANCE,
         f"variance={rec_test.feature_variance:.3f}")

    # ── [NEW-1] ActionController tests ───────────────────────────────────────
    test("T_NEW1: ActionController class exists",  ActionController is not None)
    test("T_NEW1: THREAT_LABELS defined",          len(THREAT_LABELS) >= 2)
    if action_ctrl is not None:
        test("T_NEW1: action_ctrl is ActionController", isinstance(action_ctrl, ActionController))
        # Verify trigger fires and cooldown works
        pre = action_ctrl.total_actions
        fired = action_ctrl.trigger_defense("POTENTIAL_THREAT", "test_emitter_abc", 0.95)
        test("T_NEW1: trigger_defense fires",       fired and action_ctrl.total_actions == pre + 1)
        # Second call within cooldown → suppressed
        fired2 = action_ctrl.trigger_defense("POTENTIAL_THREAT", "test_emitter_abc", 0.95)
        test("T_NEW1: cooldown suppresses repeat",  not fired2)

    # ── [NEW-2] LiveStreamSimulator tests ────────────────────────────────────
    test("T_NEW2: LiveStreamSimulator class exists", LiveStreamSimulator is not None)
    test("T_NEW2: LIVE_STREAM_DIR configured",       isinstance(LIVE_STREAM_DIR, str))
    test("T_NEW2: STREAM_INTERVAL_MS > 0",           STREAM_INTERVAL_MS > 0)
    test("T_NEW2: run_live_stream_demo callable",    callable(run_live_stream_demo))

    # ── [NEW-3] Tracker stability test ───────────────────────────────────────
    test("T_NEW3: test_tracker_stability callable",  callable(test_tracker_stability))
    ts_result = test_tracker_stability(n_obs=10, verbose=False)
    test("T_NEW3: tracker promotes after 10 obs",    ts_result)

    if TORCH_OK:
        cnn = models.get("cnn")
        test("T_A2: CNN fitted", cnn is not None and cnn.fitted)
        if CUDA_OK and cnn is not None and cnn.model is not None:
            dev = next(cnn.model.parameters()).device
            test("T_A2: CNN on CUDA", str(dev) == "cuda")

    test("T_P1: DeepSVDDDetector used",  osd_detector is not None and isinstance(osd_detector, DeepSVDDDetector))
    test("T_P1: SVDD fitted",            osd_detector is not None and osd_detector.fitted)
    test("T_P2: HysteresisFilter",       hysteresis_filter is not None)
    test("T_P3: bypass threshold",       abs(CONFIDENCE_BYPASS_THRESHOLD - 0.999999) < 1e-9)
    test("T_FIX1: _FeatureCache exists",       isinstance(_ROUTE_CACHE, _FeatureCache))
    test("T_FIX1: RF_FAST_PATH=0.97",          abs(RF_FAST_PATH_THRESHOLD - 0.97) < 1e-9)
    test("T_FIX2: DRONE_OPEN_SET_PCT=10.0",    abs(DRONE_OPEN_SET_PERCENTILE - 10.0) < 1e-9)
    test("T_FIX2: FRIENDLY_PCT=45",            FRIENDLY_PERCENTILE == 45)
    test("T_FIX2: open < decision < friendly",
         fusion.open_set_threshold < fusion.decision_threshold() < fusion.friendly_threshold)
    test("T_FIX3: preseed callable",           callable(preseed_fingerprint_db))
    stk = models.get("stacker")
    test("T_FIX4: stacker fitted",             stk is not None and stk.fitted)
    test("T_FIX4: fusion.stacker wired",       fusion.stacker is not None and fusion.stacker.fitted)

    for cls in range(3):
        fv = _generate_rf_burst(cls, rng)
        try:
            sc = fusion.score(fv)
            ok = (isinstance(sc["soft_score"], float) and 0. <= sc["soft_score"] <= 1.)
            test(f"T_SCORE cls={cls}", ok)
        except Exception as e:
            test(f"T_SCORE cls={cls}", False, str(e))

    if eval_results:
        test(f"T_BEH_RECALL ≥{GATE_RECALL_MIN:.0%}",
             eval_results.get("threat_recall",0) >= GATE_RECALL_MIN,
             f"got={eval_results.get('threat_recall',0):.1%}")
        test(f"T_BEH_HOLD ≤{GATE_HOLD_MAX:.0%}",
             eval_results.get("hold_frac",1) <= GATE_HOLD_MAX,
             f"got={eval_results.get('hold_frac',1):.1%}")
        test(f"T_BEH_OS ≥{GATE_OPEN_SET_MIN:.0%}",
             eval_results.get("open_frac",0) >= GATE_OPEN_SET_MIN,
             f"got={eval_results.get('open_frac',0):.1%}")
        test("T_BEH_FA ≤10%",
             eval_results.get("false_alarm",1) <= 0.10,
             f"got={eval_results.get('false_alarm',1):.1%}")

    if stress_results is not None:
        test("T_M4: Ghost Hunt",     stress_results.get("ghost_hunt",{}).get("pass",False))
        test("T_M4: Adversarial",    stress_results.get("adversarial",{}).get("pass",False))
        test("T_M4: Recovery Time",  stress_results.get("recovery",{}).get("pass",False))

    print(f"\n  Results: {passed} passed / {failed} failed / {passed+failed} total")
    if failed==0: print("  🎉 All tests passed — v32-FIELD consistent")
    else:         print("  ⚠️  Some tests failed — review above")
    return failed == 0


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 16 · EVALUATION
# ─────────────────────────────────────────────────────────────────────────────
class SystemMonitor:
    def __init__(self, window=MONITOR_WINDOW):
        self.window=window; self.decisions=deque(maxlen=window); self.baseline=None

    def record(self, label, soft_score, threat_score):
        self.decisions.append((label, soft_score, threat_score))
        if len(self.decisions)==self.window and self.baseline is None:
            self.baseline=float(np.mean([d[1] for d in self.decisions]))

    def report(self):
        if not self.decisions: return {}
        labels=[d[0] for d in self.decisions]; scores=[d[1] for d in self.decisions]
        n=len(labels); ctr=Counter(labels)
        hold_pct=ctr.get("HOLD",0)/n*100
        open_pct=(ctr.get("OPEN_SET_UNKNOWN",0)+ctr.get("UNKNOWN_MONITOR",0))/n*100
        fa_pct=(ctr.get("POTENTIAL_THREAT",0)+ctr.get("CONFIRMED_THREAT",0))/n*100
        mem_pct=ctr.get("MEMORY_MATCH",0)/n*100
        mean_sc=float(np.mean(scores))
        drift=float(mean_sc-self.baseline) if self.baseline else 0.
        alerts=[]
        if open_pct>50: alerts.append(f"⚠️  HIGH UNKNOWN: {open_pct:.0f}%")
        if fa_pct>10:   alerts.append(f"⚠️  HIGH FA: {fa_pct:.0f}%")
        if hold_pct>25: alerts.append(f"🚨 HOLD EXPLOSION: {hold_pct:.0f}%")
        return {"n_decisions":n,"open_pct":round(open_pct,1),"false_alarm_pct":round(fa_pct,1),
                "hold_pct":round(hold_pct,1),"memory_pct":round(mem_pct,1),
                "mean_soft_score":round(mean_sc,4),"score_drift":round(drift,4),
                "label_distribution":{k:round(v/n*100,1) for k,v in ctr.most_common()},
                "alerts":alerts}

    def print_report(self):
        r=self.report()
        if not r: return
        print(f"\n  ── MONITOR ({r['n_decisions']} decisions) ──")
        for lbl,pct in r["label_distribution"].items():
            print(f"    {DECISION_ICONS.get(lbl,'?')} {lbl:<30} {pct:>5.1f}%")
        for alert in r["alerts"]: print(f"  {alert}")


def run_full_evaluation(X_raw_te, y_te, classify_signal, classes_present, monitor):
    print(f"\n{'='*65}\nFULL EVALUATION  ({len(X_raw_te)} test samples)\n{'='*65}")
    test_decs=[]
    for i in range(len(X_raw_te)):
        dec=classify_signal(X_raw_te[i], return_bayes=True)
        dec["true_class"]=classes_present[y_te[i]]
        monitor.record(dec["label"], dec.get("soft_score",0),
                       dec.get("bayesian",{}).get("threat_score",0) if isinstance(dec.get("bayesian"),dict) else 0)
        test_decs.append(dec)
    test_df=pd.DataFrame(test_decs)

    for col in ["clf_conf","cnn_conf","evm_score","normality","ens_epistemic",
                "predictive_entropy","threat_score","soft_score","winner",
                "agreement_score","margin","sub_boost","max_clf_prob","decision_threshold"]:
        if "bayesian" in test_df.columns:
            test_df[col]=test_df["bayesian"].apply(
                lambda b: b.get(col) if isinstance(b,dict) else None)
        else:
            test_df[col]=None

    if "bypass_used" not in test_df.columns: test_df["bypass_used"]=False
    if "source"      not in test_df.columns: test_df["source"]="CLASSIFIER"

    not_detected={"POTENTIAL_THREAT","CONFIRMED_THREAT","UNKNOWN_MONITOR",
                  "SAFE_NEW_DRONE","TRUSTED_NEW_DRONE","OPEN_SET_UNKNOWN","HOLD"}
    known_mask=~test_df["label"].isin(not_detected)
    correct=((test_df.loc[known_mask,"winner"]==test_df.loc[known_mask,"true_class"]).mean()
             if known_mask.sum()>0 else 0.)
    false_alarm =test_df["label"].isin(["POTENTIAL_THREAT","CONFIRMED_THREAT"]).mean()
    bg_recall   =(test_df[test_df["true_class"]==BG_NAME]["label"].eq("BACKGROUND").mean()
                 if (test_df["true_class"]==BG_NAME).any() else 0.)
    open_frac   =float((test_df["label"]=="OPEN_SET_UNKNOWN").mean())
    hold_frac   =float((test_df["label"]=="HOLD").mean())
    bypass_frac =float(test_df["bypass_used"].fillna(False).mean())
    memory_frac =float((test_df["source"]=="MEMORY_MATCH").mean())

    labels_list =test_df["label"].tolist()
    flicker_idx =sum(1 for a,b in zip(labels_list,labels_list[1:]) if a!=b)/max(len(labels_list)-1,1)

    threat_mask    =(test_df["true_class"]!=BG_NAME)
    threat_detected=~test_df.loc[threat_mask,"label"].isin(not_detected)
    threat_recall  =float(threat_detected.mean()) if threat_mask.sum()>0 else 0.

    drone_recall_per_class={}
    for cls_name in [c for c in classes_present if c!=BG_NAME]:
        cls_mask=(test_df["true_class"]==cls_name)
        if cls_mask.sum()>0:
            detected=~test_df.loc[cls_mask,"label"].isin(not_detected)
            drone_recall_per_class[cls_name]=float(detected.mean())

    ok=lambda v,t,hi=True:"✅" if (v>=t if hi else v<=t) else "❌"
    hold_ok =("✅" if 0.045<=hold_frac<=GATE_HOLD_MAX else ("⚠️ LOW" if hold_frac<0.045 else "❌ HIGH"))
    open_ok =("✅" if GATE_OPEN_SET_MIN<=open_frac<=0.30 else ("⚠️ LOW" if open_frac<GATE_OPEN_SET_MIN else "❌ HIGH"))

    print(f"\n  ┌{'─'*74}┐")
    print(f"  │  {'METRIC':<46} {'VALUE':>8}  {'STATUS':>16}  │")
    print(f"  ├{'─'*74}┤")
    print(f"  │  {'Drone detection recall':<46} {threat_recall:>7.1%}  {ok(threat_recall,GATE_RECALL_MIN)} ≥{GATE_RECALL_MIN:.0%} ★  │")
    for cls_name,rcl in drone_recall_per_class.items():
        print(f"  │    └─ {cls_name:<41} {rcl:>7.1%}  {ok(rcl,.80)}            │")
    print(f"  │  {'False alarm rate':<46} {false_alarm:>7.1%}  {ok(false_alarm,GATE_FPR_MAX,False)} ≤{GATE_FPR_MAX:.0%}    │")
    print(f"  │  {'HOLD fraction':<46} {hold_frac:>7.1%}  {hold_ok}        │")
    print(f"  │  {'Flicker Index':<46} {flicker_idx:>7.3f}  {ok(flicker_idx,GATE_FLICKER_MAX,False)} <{GATE_FLICKER_MAX:.2f}  │")
    print(f"  │  {'Memory DB hit-rate [FIX-6]':<46} {memory_frac:>7.1%}  {'✅' if memory_frac>=GATE_HIT_RATE_MIN else '⚠️'} ≥{GATE_HIT_RATE_MIN:.0%}  │")
    print(f"  │  {'Open-set fraction':<46} {open_frac:>7.1%}  {open_ok} ≥{GATE_OPEN_SET_MIN:.0%}  │")
    print(f"  └{'─'*74}┘")

    gates=[
        (f"Integrity:  Recall ≥ {GATE_RECALL_MIN:.0%}",   threat_recall >= GATE_RECALL_MIN),
        (f"Safety:     FA ≤ {GATE_FPR_MAX:.0%}",          false_alarm   <= GATE_FPR_MAX),
        (f"Cognitive:  HOLD ≤ {GATE_HOLD_MAX:.0%}",       hold_frac     <= GATE_HOLD_MAX),
        (f"Identity:   Flicker < {GATE_FLICKER_MAX:.2f}", flicker_idx   <  GATE_FLICKER_MAX),
        (f"Memory:     OPEN_SET ≥ {GATE_OPEN_SET_MIN:.0%}", open_frac   >= GATE_OPEN_SET_MIN),
        ("Bypass:     bypass < 10%",                       bypass_frac   <  GATE_BYPASS_MAX),
    ]
    all_pass = all(v for _,v in gates)
    print(f"\n  [M3] PRODUCTION READINESS GATE:")
    for name,v in gates: print(f"    {'✅' if v else '❌'} {name}")
    if all_pass: print(f"\n  🎉 ALL GATES PASSED — PRODUCTION READY")
    else:        print(f"\n  ⚠️  SOME GATES FAILED")

    print(f"\n  Label distribution:")
    for lbl,cnt in test_df["label"].value_counts().items():
        print(f"    {DECISION_ICONS.get(lbl,'?')} {lbl:<32} {cnt:>5}  ({cnt/len(test_df):.1%})")

    test_df.to_csv("system_test_decisions_v32.csv", index=False)
    return {"test_df":test_df,"known_mask":known_mask,"correct":correct,
            "false_alarm":false_alarm,"bg_recall":bg_recall,
            "open_frac":open_frac,"hold_frac":hold_frac,"threat_recall":threat_recall,
            "bypass_frac":bypass_frac,"memory_frac":memory_frac,"flicker_idx":flicker_idx,
            "drone_recall_per_class":drone_recall_per_class,"all_gates_passed":all_pass}


def run_latency_benchmark(classify_signal, X_raw_te, n_samples=200):
    print(f"\n{'='*60}\nLATENCY BENCHMARK  (n={n_samples})\n{'='*60}")
    for i in range(20): classify_signal(X_raw_te[i % len(X_raw_te)])
    times_ms=[]
    for i in range(n_samples):
        t0=time.perf_counter(); classify_signal(X_raw_te[i % len(X_raw_te)])
        times_ms.append((time.perf_counter()-t0)*1000)
    arr=np.array(times_ms)
    stats={k:round(float(v),3) for k,v in {
        "mean_ms":arr.mean(),"p50_ms":np.percentile(arr,50),
        "p95_ms":np.percentile(arr,95),"p99_ms":np.percentile(arr,99),
        "min_ms":arr.min(),"max_ms":arr.max()}.items()}
    p95 = stats["p95_ms"]
    target_flag = ("✅ <100ms" if p95<100 else "⚠️  ≥100ms — enable CUDA or TensorRT")
    for k,v in stats.items():
        flag = f"  {target_flag}" if k=="p95_ms" else ""
        print(f"  {k:<20} {v:>10.3f} ms{flag}")
    if p95 >= 100:
        print(f"\n  [FIX-5] p95={p95:.0f}ms > 100ms target.  Options to hit <100ms:")
        print(f"    1. LGB GPU:   pip install lightgbm --install-option=--gpu")
        print(f"    2. CNN CUDA:  install torch with CUDA (auto-detects)")
        print(f"    3. TensorRT:  set EXPORT_TENSORRT=True")
    return stats


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 16b · READINESS SCORECARD
# ─────────────────────────────────────────────────────────────────────────────
def print_readiness_scorecard(eval_results, latency_stats, stress_results,
                               fp_db, tracker, action_ctrl=None):
    sep = "═" * 74
    dr   = eval_results.get("drone_recall_per_class", {})
    bp   = eval_results.get("bypass_frac", 0.)
    mem  = eval_results.get("memory_frac", 0.)
    fli  = eval_results.get("flicker_idx", 1.)
    all_gates = eval_results.get("all_gates_passed", False)

    stress_gh  = stress_results.get("ghost_hunt",  {}).get("pass", False) if stress_results else False
    stress_adv = stress_results.get("adversarial", {}).get("pass", False) if stress_results else False
    stress_rec = stress_results.get("recovery",    {}).get("pass", False) if stress_results else False
    adv_safe   = stress_results.get("adversarial", {}).get("safe_rate", 0.) if stress_results else 0.
    ttt_s      = stress_results.get("recovery",    {}).get("simulated_ttt_s") if stress_results else None
    ttt_str    = f"{ttt_s:.1f}s" if ttt_s is not None else "N/A"

    action_line = ""
    if action_ctrl is not None:
        action_line = (f"\n  [NEW-1] ActionController: "
                       f"{action_ctrl.total_actions} defense action(s) fired "
                       f"→ {action_ctrl.log_path}")

    print(f"\n{sep}")
    print("  ANTI-DRONE AI  —  v32-FIELD  READINESS SCORECARD  [M5]")
    print(f"{sep}")
    print(f"""
  System performance summary:

   ★ {eval_results.get('threat_recall',0):.0%} Drone Detection Recall   (target ≥{GATE_RECALL_MIN:.0%})""")
    for cls_name, rcl in dr.items():
        print(f"       {cls_name:<22}: {rcl:.0%}")
    print(f"""   ★ {eval_results.get('false_alarm',0):.1%} False Alarm Rate         (target ≤{GATE_FPR_MAX:.0%})
   ★ {eval_results.get('hold_frac',0):.1%} Hold / Ambiguity Rate    (target ≤{GATE_HOLD_MAX:.0%})
   ★ {fli:.3f} Flicker Index            (target <{GATE_FLICKER_MAX:.2f})
   ★ {mem:.1%} Memory DB Hit-Rate       (target ≥{GATE_HIT_RATE_MIN:.0%})
   ★ {eval_results.get('open_frac',0):.1%} Open-Set Sensitivity     (target ≥{GATE_OPEN_SET_MIN:.0%})

  Latency:  p50={latency_stats.get('p50_ms',0):.1f}ms  p95={latency_stats.get('p95_ms',0):.1f}ms  p99={latency_stats.get('p99_ms',0):.1f}ms

  [M4] Three Stress-Tests:
    {'✅' if stress_gh  else '❌'} Ghost Hunt      : {0 if stress_gh else '>0'} label transitions
    {'✅' if stress_adv else '❌'} Adversarial     : {adv_safe:.0%} noise → safe labels
    {'✅' if stress_rec else '❌'} Recovery Time   : stable at {ttt_str}

  DB / Tracker state:
    {fp_db.summary()}
    {tracker.summary()}
{action_line}

  v32 ADDITIONS vs v31-FIELD:
    [NEW-1] ActionController (Layer 3 AI-Harness)
            MockJammer → defense_log.txt + console alert
            Swap with RealHardwareController for live deployment
    [NEW-2] LiveStreamSimulator (Virtual SDR / SITL)
            run_live_stream_demo() streams bursts at {STREAM_INTERVAL_MS}ms intervals
            Plug in real DroneRF CSVs via csv_dir= argument
    [NEW-3] TemporalTracker audit logging in classify_signal
            Per-burst trust state printed + JSONL-logged
            test_tracker_stability() unit test included

  PRODUCTION STATUS:  {'🎉 ALL GATES PASSED — READY FOR DEPLOYMENT' if all_gates else '⚠️  SOME GATES FAILED — DO NOT DEPLOY'}
""")
    print(sep)


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 17 · MAIN
# ─────────────────────────────────────────────────────────────────────────────
def run_v32_main():
    print(f"\n{'█'*74}")
    print("  ANTI-DRONE AI  —  v32-FIELD")
    print(f"  [NEW-1] ActionController  [NEW-2] LiveStreamSimulator  [NEW-3] TrustAudit")
    print(f"  [FIX-5] LGB ({_LGB_DEVICE}) + CNN/SVDD CUDA={CUDA_OK} + TensorRT={EXPORT_TENSORRT}")
    print(f"  [FIX-6] TRUST_MAX_VARIANCE={TRUST_MAX_VARIANCE}  PRESEED={PRESEED_N_PER_CLASS}/class")
    print(f"{'█'*74}\n")

    # ── [NEW-3] Run standalone tracker unit test first ────────────────────────
    print("  Running [NEW-3] Tracker Stability test (standalone) ...")
    test_tracker_stability(n_obs=10, cls=1, noise_scale=0.1)

    df = build_or_load_dataset(DATA_DIR)
    X_use, y_mapped, lmap, CP, N_CLS = prepare_data(df)
    X_raw_full = X_use.copy()

    router, mi, X_master, X_rf, X_gbt, X_sub = validate_and_select_features(
        X_raw_full, y_mapped)
    _HASH_IDX[0] = router.master_idx[:HASH_TOP_FEATURES]

    M = build_and_evaluate(router, X_raw_full, y_mapped,
                            X_master, X_rf, X_gbt, X_sub, CP)

    gbp = M["gbp_for_stack"]

    print(f"\n{'='*60}\nLAPLACE APPROXIMATION\n{'='*60}")
    laplace = LaplaceApproximation().fit(M["lr"], M["X_sm_m"], M["y_sm"], N_CLS)

    print(f"\n{'='*60}\n[P1] DEEP SVDD  (device={DEVICE})\n{'='*60}")
    osd = DeepSVDDDetector().fit(M["X_sm_m"], M["y_sm"])

    print(f"\n{'='*60}\nANOMALY DETECTORS\n{'='*60}")
    det_m = MahalanobisDetector().fit(M["X_sm_m"], M["y_sm"])
    det_i = IsoForestDetector().fit(M["X_sm_m"])
    ts    = ThreatScorer(det_m, det_i, M["X_sm_m"])

    fusion = SoftFusionEngine(
        router=router, rf=M["rf"], gbt=M["gbt"], gbp=gbp,
        ens=M["ens"], cnn=M["cnn"], osd=osd, ts_det=ts,
        laplace=laplace, ts_cal=M["ts"],
        sub_clf=M["sub_clf"], classes=CP,
        open_thr=0.35, friendly_thr=0.55)
    fusion.stacker = M["stacker"]

    idx_tr, idx_te = train_test_split(
        np.arange(len(X_raw_full)), test_size=0.20, stratify=y_mapped, random_state=RANDOM_SEED)
    _, idx_val = train_test_split(
        idx_tr, test_size=0.15, stratify=y_mapped[idx_tr], random_state=RANDOM_SEED)

    X_raw_val = X_raw_full[idx_val]; X_raw_te = X_raw_full[idx_te]
    y_val_raw = y_mapped[idx_val];   y_te_raw  = y_mapped[idx_te]
    X_raw_tr  = X_raw_full[idx_tr];  y_tr_raw  = y_mapped[idx_tr]

    fusion.calibrate_thresholds_roc(X_raw_val, y_val_raw, CP)
    print(f"\n✓ Thresholds  open={fusion.open_set_threshold:.4f}  "
          f"decision={fusion.decision_threshold():.4f}  "
          f"friendly={fusion.friendly_threshold:.4f}")

    fp_db    = FingerprintDatabase(DB_PATH)
    tracker  = TemporalTracker()
    failsafe = FailSafeGuard()

    # ── [NEW-1] Instantiate ActionController ─────────────────────────────────
    action_ctrl = ActionController(log_path=DEFENSE_LOG_PATH, enabled=True)
    print(f"\n  ✓ [NEW-1] ActionController ready  → {DEFENSE_LOG_PATH}")

    _raw_classify   = make_classify_fn(
        fusion, fp_db, tracker, CP, ts, failsafe,
        action_ctrl=action_ctrl)          # [NEW-1] wired in here
    hysteresis      = HysteresisFilter(_raw_classify)
    classify_signal = hysteresis.classify

    preseed_fingerprint_db(
        fp_db, tracker, X_raw_tr, y_tr_raw,
        classify_signal, CP, n_per_class=PRESEED_N_PER_CLASS)
    hysteresis.reset()

    _ROUTE_CACHE.clear()
    latency_stats = run_latency_benchmark(classify_signal, X_raw_te)

    tracker.reset(); hysteresis.reset()
    eval_monitor = SystemMonitor()
    eval_results = run_full_evaluation(
        X_raw_te, y_te_raw, classify_signal, CP, eval_monitor)
    eval_monitor.print_report()

    total = _ROUTE_CACHE.hits + _ROUTE_CACHE.misses
    if total > 0:
        print(f"\n  [FIX-1] Route cache: "
              f"{_ROUTE_CACHE.hits}/{total} hits ({_ROUTE_CACHE.hits/total:.1%})")

    # ── [NEW-2] Live-stream demo (synthetic bursts) ───────────────────────────
    run_live_stream_demo(classify_signal, max_bursts=20, interval_ms=10)

    stress_results, stress_all_pass = run_stress_tests(
        classify_signal, fp_db, tracker, fusion, router, CP)

    run_diagnostics(M["rf"], M["X_te_rf"], M["y_te_rf"],
                    router, M["X_te_raw"], eval_results["test_df"], CP, DIAG_DIR)

    all_models = {**M, "ts": M["ts"]}
    run_self_tests(fusion, all_models, router, df, eval_results,
                   osd_detector=osd, hysteresis_filter=hysteresis,
                   stress_results=stress_results,
                   action_ctrl=action_ctrl)       # [NEW-1] passed for tests

    fp_db.save()
    json.dump(fusion.calibration_info,
              open("calibration_report_v32.json","w"), indent=2)
    print(f"✓ Calibration report → calibration_report_v32.json")

    print_readiness_scorecard(
        eval_results, latency_stats, stress_results, fp_db, tracker,
        action_ctrl=action_ctrl)

    return fusion, eval_results, latency_stats, action_ctrl


# ─────────────────────────────────────────────────────────────────────────────
# ENTRY POINT
# ─────────────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    run_v32_main()

2026/05/11 07:15:31 INFO mlflow.tracking.fluent: Autologging successfully enabled for statsmodels.
2026/05/11 07:15:47 INFO mlflow.tracking.fluent: Autologging successfully enabled for lightgbm.


✓ LightGBM available — fast inference path enabled


2026/05/11 07:15:48 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '879220ded8994dbf9f74ab1bce2b8246', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current lightgbm workflow
2026/05/11 07:15:48 WARNING mlflow.lightgbm: Failed to log dataset information to MLflow Tracking. Reason: 'list' object has no attribute 'flatten'


🏃 View run carefree-zebra-167 at: https://dagshub.com/anamitra1205/my-first-repo.mlflow/#/experiments/1/runs/879220ded8994dbf9f74ab1bce2b8246
🧪 View experiment at: https://dagshub.com/anamitra1205/my-first-repo.mlflow/#/experiments/1
  LightGBM running on CPU (no GPU or CUDA not available)
✓ PyTorch CPU — 1D-CNN + Deep SVDD enabled (no CUDA)
✓ v32-FIELD  |  Python 3.12.13
  PRODUCTION_MODE = False
  LGB device = cpu  |  CUDA = False
✓ Features: 53 RF + 18 flight + 12 comm = 83 total


DEBUG:antidrone.v32:{"ts": 1778483759.365, "event": "TRACKER_STABILITY_OBS", "obs": 1, "seen": 1, "variance": 1.0, "trust": 0.0, "trustworthy": false}
DEBUG:antidrone.v32:{"ts": 1778483759.3806, "event": "TRACKER_STABILITY_OBS", "obs": 2, "seen": 2, "variance": 0.8735, "trust": 0.0791, "trustworthy": false}
DEBUG:antidrone.v32:{"ts": 1778483759.3854, "event": "TRACKER_STABILITY_OBS", "obs": 3, "seen": 3, "variance": 868163751247872.0, "trust": 0.0, "trustworthy": false}
DEBUG:antidrone.v32:{"ts": 1778483759.4001, "event": "TRACKER_STABILITY_OBS", "obs": 4, "seen": 4, "variance": 0.8735, "trust": 0.081, "trustworthy": true}
DEBUG:antidrone.v32:{"ts": 1778483759.4126, "event": "TRACKER_STABILITY_OBS", "obs": 5, "seen": 5, "variance": 217040937811968.0, "trust": 0.0, "trustworthy": false}
DEBUG:antidrone.v32:{"ts": 1778483759.4254, "event": "TRACKER_STABILITY_OBS", "obs": 6, "seen": 6, "variance": 868163751247872.0, "trust": 0.0, "trustworthy": false}
DEBUG:antidrone.v32:{"ts": 1778483759

...

  TRUST  ID=3fae27 | Label=Phantom Drone | Seen=4 | Var=0.927 | Trust=0.00 | Trustworthy=False
  TRUST  ID=c97c06 | Label=Phantom Drone | Seen=1 | Var=1.000 | Trust=0.00 | Trustworthy=False
  TRUST  ID=3a132e | Label=Phantom Drone | Seen=3 | Var=0.922 | Trust=0.00 | Trustworthy=False


DEBUG:antidrone.v32:{"ts": 1778486271.9426, "event": "TRUST_STATE", "emitter_id": "1caf359e", "label": "Phantom Drone", "seen": 2, "variance": 0.9157, "trust": 0.0, "trustworthy": false}
DEBUG:antidrone.v32:{"ts": 1778486272.0137, "event": "TRUST_STATE", "emitter_id": "1caf359e", "label": "Phantom Drone", "seen": 3, "variance": 0.9383, "trust": 0.0, "trustworthy": false}


  TRUST  ID=1caf35 | Label=Phantom Drone | Seen=2 | Var=0.916 | Trust=0.00 | Trustworthy=False
  TRUST  ID=1caf35 | Label=Phantom Drone | Seen=3 | Var=0.938 | Trust=0.00 | Trustworthy=False
    Stable at burst #4  TTT=0.2s  p95=75.0ms
    ✅ PASS

  🎉 All stress-tests passed

DIAGNOSTICS
  ⚠  SHAP failed: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()


DEBUG:antidrone.v32:{"ts": 1778486286.0522, "event": "ACTION_TRIGGERED", "threat_label": "POTENTIAL_THREAT", "emitter_id": "test_emi", "soft_score": 0.95, "action_n": 3}
DEBUG:antidrone.v32:{"ts": 1778486286.0547, "event": "TRACKER_STABILITY_OBS", "obs": 1, "seen": 1, "variance": 1.0, "trust": 0.0, "trustworthy": false}
DEBUG:antidrone.v32:{"ts": 1778486286.0565, "event": "TRACKER_STABILITY_OBS", "obs": 2, "seen": 2, "variance": 0.8735, "trust": 0.0791, "trustworthy": false}
DEBUG:antidrone.v32:{"ts": 1778486286.0588, "event": "TRACKER_STABILITY_OBS", "obs": 3, "seen": 3, "variance": 868163751247872.0, "trust": 0.0, "trustworthy": false}
DEBUG:antidrone.v32:{"ts": 1778486286.0607, "event": "TRACKER_STABILITY_OBS", "obs": 4, "seen": 4, "variance": 0.8735, "trust": 0.081, "trustworthy": true}
DEBUG:antidrone.v32:{"ts": 1778486286.0645, "event": "TRACKER_STABILITY_OBS", "obs": 5, "seen": 5, "variance": 217040937811968.0, "trust": 0.0, "trustworthy": false}
DEBUG:antidrone.v32:{"ts": 17784

  ✓ Calibration curves → diagnostics_v32/calibration_curves.png

SELF-TEST SUITE  (v32-FIELD)
  ✅ PASS  T1:  N_FEATURES=83
  ✅ PASS  T1b: HLBR in schema
  ✅ PASS  T2a: RF shape
  ✅ PASS  T2b: GBT shape
  ✅ PASS  T3a: RF predict_proba
  ✅ PASS  T4:  Temperature in range
  ✅ PASS  T_FIX5: LGB_OK or sklearn fallback
  ✅ PASS  T_FIX5: LGB_DEVICE defined
  ✅ PASS  T_FIX5: RF is LGBClassifier
  ✅ PASS  T_FIX5: RF booster fitted
  ✅ PASS  T_FIX5: GBT is LGBClassifier
  ✅ PASS  T_FIX5: CUDA flag defined
  ✅ PASS  T_FIX6: TRUST_MAX_VARIANCE=0.90
  ✅ PASS  T_FIX6: PRESEED_N_PER_CLASS=80
  ❌ FAIL  T_FIX6: real-world variance passes is_trustworthy  variance=0.904
  ✅ PASS  T_NEW1: ActionController class exists
  ✅ PASS  T_NEW1: THREAT_LABELS defined
  ✅ PASS  T_NEW1: action_ctrl is ActionController
  📡 SENT SIGNAL TO JAMMER: POTENTIAL_THREAT [emitter=test_emi  score=0.9500]
  ✅ PASS  T_NEW1: trigger_defense fires
  ✅ PASS  T_NEW1: cooldown suppresses repeat
  ✅ PASS  T_NEW2: LiveStreamSimulator cl